# Backpressure and Scalability

## Overview

This notebook studies what happens when profiling events are generated faster than the downstream pipeline can process, transmit, or store them.

The previous notebooks established the basic profiling data path:

```text
ProfileEvent
    ↓
BoundedQueue
    ↓
Serialization
    ↓
Framing
    ↓
Batching
    ↓
TCP
    ↓
Collector
```

So far, most experiments assumed that the downstream stages were fast enough to keep up with event production.

In a real profiling system, this assumption may fail.

For example:

```text
Producer rate:
100,000 events / second

Consumer rate:
30,000 events / second
```

If the system uses an unbounded queue, events continue accumulating:

```text
Producer
   ↓
Queue
[events]
[more events]
[more events]
[more events]
   ↓
Consumer
```

This can cause:

```text
increasing memory usage
high queueing latency
event backlog
eventual out-of-memory failure
```

To prevent this, the profiling pipeline needs a controlled overload strategy.

---

## Core Question

The central question of this notebook is:

> What should a profiling system do when events are generated faster than they can be consumed?

We will study several possible answers:

```text
Block the producer

Drop the newest event

Drop the oldest event

Increase batching efficiency
```

Each approach has different tradeoffs.

---

## Backpressure

Backpressure occurs when a downstream component cannot process data as quickly as an upstream component produces it.

For example:

```text
Fast Producer
      ↓
Bounded Queue
████████████ FULL
      ↓
Slow Consumer
```

Once the queue becomes full, the system must react.

With a blocking queue:

```text
queue full
    ↓
producer waits
    ↓
consumer removes an event
    ↓
space becomes available
    ↓
producer continues
```

The slowdown therefore propagates backward through the pipeline.

This is called:

```text
backpressure propagation
```

A larger system may look like:

```text
Slow Collector
      ↓
TCP Sender slows down
      ↓
Consumer slows down
      ↓
Queue fills
      ↓
Producer blocks or drops events
```

---

## Why Backpressure Matters for Profiling

A profiler should observe an application while disturbing it as little as possible.

If producing a profiling event can block the application thread:

```text
Application
    ↓
record profiling event
    ↓
queue full
    ↓
wait
```

then the profiler itself can change the application's execution time.

For example:

```text
Without profiler:
100 ms

With heavy profiler backpressure:
150 ms
```

This introduces:

```text
profiling overhead
```

and can make performance measurements less representative of the original application.

Therefore, profiling infrastructure must balance:

```text
data completeness
        vs
profiling overhead
        vs
memory usage
        vs
latency
```

---

# Notebook Plan

This notebook is divided into eight sections.

---

## 1. Standalone Setup and Metrics Design

We will recreate only the components required for the experiments:

```text
ProfileEvent
BoundedQueue
producer thread
consumer thread
timing utilities
```

We will also define the metrics used throughout the notebook.

Examples include:

```text
events produced
events consumed
events dropped

producer throughput
consumer throughput

queue occupancy
maximum queue occupancy

producer blocked time

total runtime
```

The goal is to make backpressure measurable rather than only observable.

---

## 2. Fast Producer and Slow Consumer Baseline

We will intentionally create an overloaded pipeline.

For example:

```text
Producer:
very fast

Queue:
capacity = 8

Consumer:
10 ms processing delay per event
```

The expected behavior is:

```text
Producer produces quickly
        ↓
Queue fills
        ↓
Queue becomes full
        ↓
Producer blocks
        ↓
Consumer slowly releases space
```

This experiment provides the baseline backpressure behavior.

---

## 3. Measure Queue Occupancy and Producer Blocking

The basic experiment will be extended with instrumentation.

We want to measure:

```text
maximum queue occupancy

number of times producer encounters backpressure

total producer blocked time

average blocked time

consumer throughput
```

For example:

```text
Experiment runtime:
10 seconds

Producer blocked:
7.8 seconds
```

This tells us that the producer spends most of its time waiting for downstream processing.

---

## 4. Queue Capacity Experiment

We will compare several queue capacities:

```text
1
8
64
256
1024
```

The purpose is to study the relationship between:

```text
queue capacity
    ↓
memory usage
    ↓
producer blocking
    ↓
queueing latency
```

A very small queue may cause frequent blocking:

```text
small queue
    ↓
low memory usage
    ↓
frequent producer stalls
```

A larger queue may reduce short-term blocking:

```text
large queue
    ↓
more buffering
    ↓
less immediate producer blocking
```

but may increase:

```text
memory usage
event queueing latency
```

Therefore:

```text
larger queue != always better
```

---

## 5. Compare Overload Policies

We will compare three queue-overload strategies.

### Block Producer

```text
Queue full
    ↓
Producer waits
```

Advantages:

```text
no event loss
complete profiling data
```

Disadvantages:

```text
producer may stall
application interference may increase
```

---

### Drop Newest

```text
Queue:

[A][B][C][D]

New event:
E

Queue full
    ↓
drop E
```

Advantages:

```text
producer does not block
lower application interference
```

Disadvantages:

```text
profiling events are lost
```

---

### Drop Oldest

```text
Queue:

[A][B][C][D]

New event:
E

drop A
    ↓
[B][C][D][E]
```

This keeps the newest information at the cost of older events.

We will compare:

```text
throughput
drop rate
producer blocking
runtime
```

---

## 6. Add Batching to the Consumer

The consumer will then process several events together.

Instead of:

```text
pop event
    ↓
process event
```

we move toward:

```text
pop multiple events
        ↓
build batch
        ↓
process/send batch
```

For example:

```text
BoundedQueue
     ↓
pop 16 events
     ↓
BatchBuilder
     ↓
one downstream operation
```

We will compare different batch sizes:

```text
1
4
16
64
```

The expected tradeoff is:

```text
larger batch
    ↓
fewer downstream operations
    ↓
better throughput
```

but potentially:

```text
larger batch
    ↓
events wait longer
    ↓
higher latency
```

This introduces the classic:

```text
throughput vs latency
```

tradeoff.

---

## 7. Simulate Slow Network or Collector

We will artificially slow the downstream stage to simulate:

```text
network congestion
slow TCP transmission
slow collector
slow database writes
slow analysis pipeline
```

For example, the consumer may deliberately sleep after processing a batch.

This allows us to observe:

```text
Slow downstream
      ↓
Consumer throughput decreases
      ↓
Queue occupancy rises
      ↓
Backpressure reaches producer
```

This experiment demonstrates backpressure propagation across the pipeline.

---

## 8. Final Scalability Benchmark

Finally, we will compare several configurations.

Parameters may include:

```text
queue capacity

batch size

consumer delay

overload policy
```

The results can be summarized in a table such as:

| Policy | Queue Capacity | Batch Size | Throughput | Drop Rate | Producer Block Time |
|---|---:|---:|---:|---:|---:|
| Block | 8 | 1 | ... | 0% | ... |
| Block | 64 | 16 | ... | 0% | ... |
| Drop Newest | 64 | 16 | ... | ... | 0 |
| Drop Oldest | 64 | 16 | ... | ... | 0 |

The goal is not to find one universally correct configuration.

The goal is to understand the tradeoffs and explain why one configuration may be appropriate for a particular profiling workload.

---

# Overall Experimental Flow

```text
Fast Producer
      ↓
BoundedQueue
      ↓
Slow Consumer
      ↓
Observe backpressure
      ↓
Measure blocking and occupancy
      ↓
Change queue capacity
      ↓
Compare overload policies
      ↓
Add batching
      ↓
Simulate slow downstream
      ↓
Run scalability benchmark
      ↓
Analyze system tradeoffs
```

---

# Key Design Tradeoffs

Throughout the notebook, we will repeatedly evaluate four competing objectives:

```text
Data completeness
        ↕
Application interference

Memory usage
        ↕
Queueing capacity

Throughput
        ↕
Latency

Backpressure
        ↕
Event dropping
```

A profiling system must choose its behavior based on the importance of each objective.

---

# Interview-Relevant Questions

By the end of this notebook, the following questions should be answerable:

- What is backpressure?
- Why is an unbounded queue dangerous?
- How does a bounded queue control memory growth?
- What happens when the producer is faster than the consumer?
- How can producer blocking affect profiling accuracy?
- How would you measure backpressure?
- How does queue capacity affect blocking and latency?
- What are the tradeoffs between blocking and dropping events?
- What is the difference between drop-newest and drop-oldest policies?
- How can batching improve throughput?
- Why can larger batches increase latency?
- How does downstream slowdown propagate upstream?
- How would you design an overload policy for a production profiling system?

---

# Final Goal

By the end of this notebook, the profiling pipeline should evolve from:

```text
Producer
    ↓
Queue
    ↓
Consumer
```

into a measurable and configurable system:

```text
                    metrics
                       ↑
                       │
Producer
    ↓
Overload Policy
    ↓
BoundedQueue
    ↓
Batching Consumer
    ↓
Simulated Network / Collector
    ↓
Performance Analysis
```

The final objective is to be able to explain not only how the pipeline works, but also how it behaves under overload and why particular scalability decisions were made.

## Standalone Setup + Metrics Design

### Goal

The goal of this section is to build a small, self-contained environment for studying backpressure.

We do **not** add TCP, serialization, framing, or storage yet.

We first isolate the producer-consumer system:

```text
Producer
    ↓
BoundedQueue
    ↓
Consumer
```

and make its behavior measurable.

The main question is:

> When the producer is faster than the consumer, how much backpressure is created and how can we quantify it?

---

## Components Used

For the first experiment, we only need:

```text
ProfileEvent
BoundedQueue<ProfileEvent>
producer thread
consumer thread
timing utilities
metrics counters
```

We intentionally keep the system small so that any performance effect can be attributed mainly to:

```text
producer rate
consumer rate
queue capacity
```

rather than to networking or serialization overhead.

---

## Experimental Architecture

The initial experiment will use:

```text
Fast Producer
      ↓
BoundedQueue
      ↓
Slow Consumer
```

The producer generates events as quickly as possible.

The consumer intentionally processes them more slowly.

For example:

```text
Producer:
no artificial delay

Consumer:
sleep 5 ms per event
```

with:

```text
queue capacity = 8
```

This should cause the queue to fill quickly.

---

## Expected Backpressure Behavior

At the beginning:

```text
Producer
    ↓
[ ][ ][ ][ ][ ][ ][ ][ ]
    ↓
Consumer
```

The producer starts filling the queue:

```text
Producer
    ↓
[A][B][C][D][E][F][ ][ ]
    ↓
Consumer
```

Eventually:

```text
Producer
    ↓
[A][B][C][D][E][F][G][H]
                         ↑
                        FULL
    ↓
Consumer
```

Once the queue is full:

```text
producer push()
      ↓
wait
      ↓
consumer removes one event
      ↓
space becomes available
      ↓
producer continues
```

This is the simplest form of backpressure.

---

# Metrics Design

To understand the behavior of the system, we need more than just program output.

We need quantitative metrics.

The main metrics used in this notebook will be:

```text
events produced
events consumed
events dropped
queue occupancy
maximum queue occupancy
producer blocking count
producer blocked time
producer throughput
consumer throughput
total runtime
```

---

## Events Produced

This metric counts how many events the producer successfully generates.

Conceptually:

```text
producer creates event
        ↓
produced_events++
```

For the blocking policy, this will usually equal the number of events eventually inserted into the queue.

Example:

```text
Produced events: 1000
```

---

## Events Consumed

This counts how many events are successfully processed by the consumer.

Conceptually:

```text
consumer pops event
        ↓
process event
        ↓
consumed_events++
```

For a system that does not drop events, we expect:

```text
produced_events == consumed_events
```

after the pipeline finishes.

---

## Events Dropped

Later, when we introduce overload policies such as:

```text
drop newest
drop oldest
```

we will count how many events are discarded.

For the first blocking experiment:

```text
dropped_events = 0
```

because the producer waits instead of dropping data.

---

## Queue Occupancy

Queue occupancy means:

```text
number of events currently stored in the queue
```

For example:

```text
capacity = 8

current queue:

[A][B][C][D][ ][ ][ ][ ]

occupancy = 4
```

Queue occupancy gives us a direct view of backpressure.

If the queue frequently stays near capacity:

```text
occupancy ≈ capacity
```

then the consumer is not keeping up with the producer.

---

## Maximum Queue Occupancy

We will also record:

```text
max_queue_occupancy
```

Example:

```text
capacity = 64

maximum observed occupancy = 64
```

This means the queue became completely full at least once.

If:

```text
maximum observed occupancy = 7
```

then a capacity of 64 may be much larger than necessary for that workload.

---

## Producer Blocking Count

For the blocking queue policy, we want to know how often the producer encounters a full queue.

Conceptually:

```text
producer wants to push
        ↓
queue full?
        ↓
yes
        ↓
producer blocks
```

Each such occurrence can increment:

```text
producer_block_count
```

This tells us how frequently backpressure reaches the producer.

---

## Producer Blocked Time

This is one of the most important metrics.

For every `push()` operation, we can measure:

```text
start time
    ↓
queue.push(...)
    ↓
end time
```

If the queue has free space, `push()` returns quickly.

If the queue is full, `push()` may wait.

Therefore:

```text
push duration
≈
queue waiting time
+
small insertion overhead
```

The total blocked time can be accumulated across all push operations.

For example:

```text
Total runtime:
5.0 seconds

Producer blocked time:
4.1 seconds
```

This indicates very strong backpressure.

---

## Producer Throughput

Producer throughput can be defined as:

```text
produced events
----------------
total runtime
```

For example:

```text
10,000 events
--------------
2 seconds

= 5,000 events/second
```

This tells us the effective event-generation rate after accounting for blocking.

---

## Consumer Throughput

Consumer throughput is:

```text
consumed events
----------------
total runtime
```

For a stable blocking pipeline, producer and consumer throughput should eventually become similar because the slower stage controls the overall system throughput.

For example:

```text
Producer capability:
100,000 events/s

Consumer capability:
200 events/s
```

With a bounded blocking queue, long-term pipeline throughput will be much closer to:

```text
200 events/s
```

because the consumer is the bottleneck.

---

## Total Runtime

We will measure the complete experiment using:

```cpp
std::chrono::steady_clock
```

Conceptually:

```text
experiment start
      ↓
producer + consumer run
      ↓
threads finish
      ↓
experiment end
```

Then:

```text
runtime = end - start
```

`steady_clock` is appropriate for performance measurement because it is monotonic and is not affected by wall-clock adjustments.

---

# Initial Metrics Structure

Conceptually, the experiment will maintain something like:

```text
Metrics
│
├── produced_events
├── consumed_events
├── dropped_events
├── max_queue_occupancy
├── producer_block_count
├── producer_blocked_time
└── total_runtime
```

From these raw values, we can calculate:

```text
producer throughput

consumer throughput

drop rate

percentage of runtime blocked
```

For example:

```text
blocked_percentage =
producer_blocked_time
----------------------
total_runtime
× 100%
```

---

# Why These Metrics Matter

Suppose two queue configurations produce the same throughput:

```text
Queue capacity = 8

throughput = 1000 events/s
producer blocked = 80% of runtime
```

and:

```text
Queue capacity = 64

throughput = 1000 events/s
producer blocked = 30% of runtime
```

The throughput alone does not tell the whole story.

The second configuration may create less immediate pressure on the producer.

However, it may also create:

```text
more buffered events
more memory usage
higher queueing latency
```

Therefore, scalability cannot be evaluated using only one metric.

---

# First Experiment Configuration

For the baseline experiment, we will start with a deliberately overloaded configuration:

```text
Number of events:
about 100–1000

Queue capacity:
8

Producer delay:
0 ms

Consumer delay:
5–10 ms per event
```

The exact values can be adjusted after the first run.

We expect:

```text
queue fills quickly

producer frequently blocks

consumer becomes the bottleneck

no events are dropped
```

---

# Expected Result

Conceptually:

```text
Producer starts
      ↓
produces very quickly
      ↓
queue reaches capacity
      ↓
producer blocks
      ↓
consumer processes one event
      ↓
producer inserts another event
      ↓
queue stays close to full
      ↓
repeat
```

The metrics should therefore show something like:

```text
Produced events:        100
Consumed events:        100
Dropped events:           0

Max queue occupancy:      8

Producer block count:    high
Producer blocked time:   high

Producer throughput:     close to consumer throughput
Consumer throughput:     limited by artificial delay
```

---

# What We Are NOT Testing Yet

This section intentionally does not include:

```text
serialization
framing
batching
TCP
database storage
GPU profiling
```

Those components would make the experiment harder to interpret.

The goal here is to isolate:

```text
producer
    ↓
queue
    ↓
consumer
```

and understand backpressure behavior clearly before adding more system components.

---

# Questions to Answer After This Section

By the end of this section, the following questions should be answerable:

- What metrics are useful for measuring backpressure?
- Why is queue occupancy important?
- What does maximum queue occupancy tell us?
- How can producer blocked time be measured?
- Why can producer throughput become limited by consumer throughput?
- Why is `std::chrono::steady_clock` appropriate for performance measurement?
- Why is throughput alone insufficient to evaluate a queue design?
- Why should networking and serialization be excluded from the first backpressure experiment?
- What result would indicate that the consumer is the bottleneck?

In [1]:
!apt-get update -qq
!apt-get install -y -qq build-essential g++ cmake ninja-build git

!mkdir -p distributed-profiler/include
!mkdir -p distributed-profiler/src
!mkdir -p distributed-profiler/tests
!mkdir -p distributed-profiler/benchmarks
!mkdir -p distributed-profiler/apps

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ninja-build.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.11.1-2_amd64.deb ...
Unpacking ninja-build (1.11.1-2) ...
Setting up ninja-build (1.11.1-2) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
%%writefile distributed-profiler/include/profile_event.hpp

#pragma once

#include <cstdint>
#include <string>
#include <variant>

struct KernelEvent {
    std::uint64_t timestamp_ns;
    std::uint64_t duration_ns;
    std::uint32_t device_id;
    std::string kernel_name;
};

struct MemoryEvent {
    std::uint64_t timestamp_ns;
    std::uint32_t device_id;
    std::uint64_t bytes;
    std::string operation;
};

using ProfileEvent =
    std::variant<KernelEvent, MemoryEvent>;

Writing distributed-profiler/include/profile_event.hpp


In [3]:
%%writefile distributed-profiler/include/bounded_queue.hpp

#pragma once

#include <condition_variable>
#include <cstddef>
#include <deque>
#include <mutex>
#include <optional>
#include <stop_token>
#include <utility>

template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity)
        : capacity_(capacity),
          closed_(false) {}

    bool push(
        T item,
        std::stop_token stop_token
    ) {
        std::unique_lock<std::mutex> lock(mutex_);

        bool ready = not_full_.wait(
            lock,
            stop_token,
            [this]() {
                return queue_.size() < capacity_
                    || closed_;
            }
        );

        if (
            !ready ||
            stop_token.stop_requested() ||
            closed_
        ) {
            return false;
        }

        queue_.push_back(std::move(item));

        lock.unlock();
        not_empty_.notify_one();

        return true;
    }


    std::optional<T> pop(
        std::stop_token stop_token
    ) {
        std::unique_lock<std::mutex> lock(mutex_);

        bool ready = not_empty_.wait(
            lock,
            stop_token,
            [this]() {
                return !queue_.empty()
                    || closed_;
            }
        );

        if (
            !ready ||
            stop_token.stop_requested()
        ) {
            return std::nullopt;
        }

        if (
            queue_.empty() &&
            closed_
        ) {
            return std::nullopt;
        }

        T item =
            std::move(queue_.front());

        queue_.pop_front();

        lock.unlock();
        not_full_.notify_one();

        return item;
    }


    void close() {
        {
            std::lock_guard<std::mutex> lock(mutex_);
            closed_ = true;
        }

        not_empty_.notify_all();
        not_full_.notify_all();
    }


private:
    std::deque<T> queue_;
    std::size_t capacity_;

    std::mutex mutex_;

    std::condition_variable_any not_empty_;
    std::condition_variable_any not_full_;

    bool closed_;
};

Writing distributed-profiler/include/bounded_queue.hpp


In [4]:
%%writefile distributed-profiler/include/pipeline_metrics.hpp

#pragma once

#include <chrono>
#include <cstddef>
#include <cstdint>

struct PipelineMetrics {

    // ============================================================
    // Workload metrics
    // ============================================================

    // Number of events the producer attempted to generate.
    std::uint64_t attempted_events = 0;

    // Number of events successfully admitted into the pipeline.
    std::uint64_t produced_events = 0;

    // Number of events successfully consumed.
    std::uint64_t consumed_events = 0;

    // Number of events dropped.
    //
    // For the initial blocking policy, this should stay at 0.
    std::uint64_t dropped_events = 0;


    // ============================================================
    // Backpressure metrics
    // ============================================================

    // Number of times the producer encounters backpressure.
    std::uint64_t producer_block_count = 0;

    // Total time spent waiting while trying to push events.
    std::chrono::nanoseconds producer_blocked_time{0};


    // ============================================================
    // Queue metrics
    // ============================================================

    // Maximum number of events observed in the queue.
    std::size_t max_queue_occupancy = 0;


    // ============================================================
    // Experiment timing
    // ============================================================

    // Total runtime of the experiment.
    std::chrono::nanoseconds total_runtime{0};
};

Writing distributed-profiler/include/pipeline_metrics.hpp


In [5]:
%%writefile distributed-profiler/include/bounded_queue.hpp

#pragma once

#include <condition_variable>
#include <cstddef>
#include <deque>
#include <mutex>
#include <optional>
#include <stop_token>
#include <utility>

template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity)
        : capacity_(capacity),
          closed_(false) {}


    // ============================================================
    // Push one item into the queue.
    //
    // If the queue is full, wait until:
    //
    // - space becomes available
    // - queue is closed
    // - stop is requested
    // ============================================================

    bool push(
        T item,
        std::stop_token stop_token
    ) {
        // Create a unique_lock for mutex_.
        std::unique_lock<std::mutex> lock(mutex_);

        // Wait on not_full_.
        bool ready = not_full_.wait(
            lock,
            stop_token,
            [this]() {
                return queue_.size() < capacity_
                    || closed_;
            }
        );


        if (
            !ready ||
            stop_token.stop_requested() ||
            closed_
        ) {
            return false;
        }


        // Move item into queue_.
        queue_.push_back(std::move(item));

        // Unlock mutex before notification.
        lock.unlock();

        // Notify one waiting consumer.
        not_empty_.notify_one();

        // Return true.
        return true;
    }


    // ============================================================
    // Pop one item from the queue.
    //
    // If the queue is empty, wait until:
    //
    // - an item becomes available
    // - queue is closed
    // - stop is requested
    // ============================================================

    std::optional<T> pop(
        std::stop_token stop_token
    ) {
        // Create a unique_lock for mutex_.
        std::unique_lock<std::mutex> lock(mutex_);

        bool ready = not_empty_.wait(
            lock,
            stop_token,
            [this]() {
                return !queue_.empty()
                    || closed_;
            }
        );


        // If wait is interrupted or stop requested:
        //
        // return std::nullopt.
        if (
            !ready ||
            stop_token.stop_requested()
        ) {
            return std::nullopt;
        }

        // If queue is empty AND closed:
        //
        // return std::nullopt.
        if (
            queue_.empty() &&
            closed_
        ) {
            return std::nullopt;
        }


        // Move the front item into a local variable.
        T item = std::move(queue_.front());


        // Remove the front item from queue_.
        queue_.pop_front();

        // Unlock mutex.
        lock.unlock();


        // Notify one waiting producer.
        not_full_.notify_one();

        // Return the item.
        return item;
    }


    // ============================================================
    // Close the queue.
    //
    // After close():
    //
    // - no new items can be pushed
    // - blocked producers wake up
    // - blocked consumers wake up
    // - consumers can still drain remaining items
    // ============================================================

    void close() {
        {
            std::lock_guard<std::mutex> lock(mutex_);

            closed_ = true;
        }

        not_empty_.notify_all();
        not_full_.notify_all();
    }


    // ============================================================
    // Queue inspection
    // queue_ may be modified by producer/consumer threads,
    // so mutex_ must be locked before reading queue_.size().
    // ============================================================

    // Return the current number of items in the queue.
    //
    // Because another thread may modify queue_,
    // mutex_ must be locked before reading queue_.size().
    //
    // Note:
    // This is a const function.
    std::size_t size() const {
        std::lock_guard<std::mutex> lock(mutex_);
        return queue_.size();
    }


    // Return the configured queue capacity.
    //
    // capacity_ does not change after construction.
    std::size_t capacity() const {
        return capacity_;
    }



private:
    // Stores queue elements.
    std::deque<T> queue_;


    // Maximum number of elements allowed.
    std::size_t capacity_;

    mutable std::mutex mutex_;


    std::condition_variable_any not_empty_;

    std::condition_variable_any not_full_;

    bool closed_;

};

Overwriting distributed-profiler/include/bounded_queue.hpp


## Fast Producer + Slow Consumer Baseline

### Goal

The goal of this section is to create the first controlled backpressure experiment.

We intentionally make the producer much faster than the consumer:

```text
Fast Producer
      ↓
BoundedQueue
      ↓
Slow Consumer
```

The purpose is to observe how a bounded queue behaves when downstream processing cannot keep up with event production.

---

## Experimental Configuration

We will start with a simple workload:

```text
Total events:
100

Queue capacity:
8

Producer delay:
0 ms

Consumer delay:
10 ms per event
```

The producer generates events as quickly as possible.

The consumer intentionally processes each event slowly.

This should cause the queue to fill quickly.

---

## Expected Behavior

At the beginning, the producer starts inserting events:

```text
Producer
    ↓
[A][B][C][D][ ][ ][ ][ ]
    ↓
Consumer
```

Because the producer is much faster than the consumer, the queue continues filling:

```text
Producer
    ↓
[A][B][C][D][E][F][G][ ]
    ↓
Consumer
```

Eventually the queue becomes full:

```text
Producer
    ↓
[A][B][C][D][E][F][G][H]
                         ↑
                        FULL
    ↓
Consumer
```

At this point, another call to:

```cpp
queue.push(...)
```

cannot immediately insert a new event.

The producer therefore waits inside the bounded queue.

Conceptually:

```text
Producer wants to push
        ↓
Queue full
        ↓
Producer waits
        ↓
Consumer pops one event
        ↓
Queue has free space
        ↓
Producer wakes up
        ↓
Producer pushes next event
```

This is the first direct demonstration of backpressure.

---

## Producer Responsibilities

The producer thread will:

```text
generate event
    ↓
increment attempted_events
    ↓
measure push start time
    ↓
queue.push(...)
    ↓
measure push end time
    ↓
increment produced_events
    ↓
update blocked-time measurement
    ↓
inspect queue occupancy
    ↓
update max_queue_occupancy
```

The producer does not intentionally sleep.

Therefore, its natural behavior is to push events as quickly as possible.

---

## Consumer Responsibilities

The consumer thread will:

```text
queue.pop()
    ↓
receive one event
    ↓
sleep for 10 ms
    ↓
increment consumed_events
    ↓
repeat
```

The artificial sleep represents slower downstream processing.

Later, the same type of delay can represent:

```text
serialization cost
network transmission
slow collector
database writes
analysis overhead
```

For this first experiment, however, we use only a simple sleep so that the bottleneck is easy to understand.

---

## Measuring Push Duration

Before calling:

```cpp
queue.push(...)
```

the producer records:

```cpp
auto push_start = Clock::now();
```

After the push returns:

```cpp
auto push_end = Clock::now();
```

Then:

```text
push_duration
=
push_end - push_start
```

If the queue is not full, this duration should be very small.

If the queue is full:

```text
push()
    ↓
wait
    ↓
consumer removes event
    ↓
push continues
```

then the duration becomes much larger.

For the baseline experiment, we use:

```text
push duration
≈
producer blocked time
```

This is an approximation because the measurement also includes:

```text
mutex acquisition
queue insertion
small function overhead
```

Later, we can instrument the queue internally to measure the actual waiting time more precisely.

---

## Measuring Queue Occupancy

After a successful push, we can inspect:

```cpp
queue.size()
```

and compare it with the previous maximum.

Conceptually:

```text
current_size = queue.size()

if current_size > max_queue_occupancy:
    max_queue_occupancy = current_size
```

For a queue with:

```text
capacity = 8
```

if we observe:

```text
max_queue_occupancy = 8
```

then the queue reached full capacity.

This indicates strong backpressure.

---

## Thread Lifecycle

The experiment uses two worker threads:

```text
Main Thread
│
├── Producer Thread
│
└── Consumer Thread
```

The producer generates all events and then closes the queue:

```text
produce all events
      ↓
queue.close()
```

Closing the queue means:

```text
no more events will be produced
```

but the consumer is still allowed to drain remaining items.

The consumer continues until:

```text
queue is empty
AND
queue is closed
```

Then:

```cpp
queue.pop(...)
```

returns:

```cpp
std::nullopt
```

and the consumer exits.

---

## Expected Thread Sequence

Conceptually:

```text
Main
  ↓
start producer
  ↓
start consumer

Producer                Consumer
   │                       │
   ├─ push                 ├─ pop
   ├─ push                 ├─ sleep
   ├─ push                 │
   ├─ push                 │
   ├─ queue full           │
   ├─ wait                 │
   │                       ├─ pop
   ├─ wake                 ├─ sleep
   ├─ push                 │
   │                       │
   ...                     ...
   │                       │
   ├─ finish producing     │
   └─ queue.close()        │
                           ├─ drain remaining events
                           └─ exit
```

The main thread waits for both worker threads to finish.

---

## Metrics Collected

The baseline experiment will collect:

```text
attempted_events
produced_events
consumed_events
dropped_events

max_queue_occupancy

producer_blocked_time

total_runtime
```

The blocking policy should produce:

```text
dropped_events = 0
```

because the producer waits instead of discarding events.

---

## Derived Metrics

After the experiment finishes, we calculate several derived metrics.

### Producer Throughput

```text
producer_throughput
=
produced_events
---------------
total_runtime
```

Unit:

```text
events / second
```

---

### Consumer Throughput

```text
consumer_throughput
=
consumed_events
---------------
total_runtime
```

Because the consumer is intentionally slow, it should determine the long-term pipeline throughput.

---

### Maximum Queue Utilization

```text
max_queue_utilization
=
max_queue_occupancy
-------------------
queue_capacity
× 100%
```

For example:

```text
max occupancy = 8
capacity = 8

utilization = 100%
```

This indicates that the queue became completely full.

---

### Producer Blocked Percentage

```text
blocked_percentage
=
producer_blocked_time
---------------------
total_runtime
× 100%
```

For example:

```text
runtime = 1.0 s

producer blocked time = 0.85 s

blocked percentage = 85%
```

This means the producer spends most of its execution time waiting for the consumer.

---

## Expected Result

Because the consumer sleeps:

```text
10 ms / event
```

its theoretical processing rate is roughly:

```text
100 events / second
```

The producer itself may be capable of generating events much faster than this.

However, after the queue becomes full, backpressure forces the producer to slow down.

Therefore, the effective producer throughput should eventually become close to the consumer throughput.

Conceptually:

```text
Producer raw capability
very high
      ↓
queue becomes full
      ↓
producer blocked
      ↓
effective producer throughput
≈
consumer throughput
```

---

## What This Experiment Demonstrates

The experiment demonstrates an important property of bounded pipelines:

```text
the slowest stage controls long-term throughput
```

A bounded queue prevents:

```text
unbounded memory growth
```

but does so by pushing pressure backward toward the producer.

The complete behavior is:

```text
Slow Consumer
      ↓
Queue fills
      ↓
Producer blocks
      ↓
Producer throughput decreases
```

This is backpressure propagation in its simplest form.

---

## Questions to Answer After This Experiment

After completing the baseline experiment, the following questions should be answerable:

- Why does the queue become full?
- Why does the producer block?
- Why does the consumer become the bottleneck?
- Why does producer throughput approach consumer throughput?
- How does a bounded queue prevent unlimited memory growth?
- What does `max_queue_occupancy == queue_capacity` indicate?
- How can push duration approximate producer blocked time?
- Why is this only an approximation?
- Why does closing the queue not immediately stop the consumer?
- Why should the consumer drain remaining events after `close()`?
- How does this experiment demonstrate backpressure propagation?

In [6]:
%%writefile distributed-profiler/benchmarks/backpressure_baseline.cpp

#include "bounded_queue.hpp"
#include "pipeline_metrics.hpp"
#include "profile_event.hpp"

#include <algorithm>
#include <chrono>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <stop_token>
#include <thread>


int main() {

    using Clock = std::chrono::steady_clock;


    // ============================================================
    // Experiment configuration
    // ============================================================

    const std::size_t total_events = 100;

    const std::size_t queue_capacity = 8;

    const auto consumer_delay =
        std::chrono::milliseconds(10);


    // ============================================================
    // Queue and metrics
    // ============================================================

    BoundedQueue<ProfileEvent> queue(queue_capacity);

    PipelineMetrics metrics;


    // ============================================================
    // Experiment start time
    // ============================================================

    auto experiment_start = Clock::now();


    // ============================================================
    // Producer thread
    //
    // create event
    //      ↓
    // queue.push()
    //      ↓
    // measure push duration
    //      ↓
    // inspect queue occupancy
    // ============================================================

    std::jthread producer(
        [&](std::stop_token stop_token) {

            for (
                std::size_t i = 0;
                i < total_events;
                ++i
            ) {
                // Producer attempts to generate one event.
                ++metrics.attempted_events;


                // Create a synthetic profiling event.
                ProfileEvent event =
                    KernelEvent{
                        static_cast<std::uint64_t>(i),
                        100,
                        0,
                        "synthetic_kernel"
                    };


                // Measure how long push() takes.
                auto push_start = Clock::now();

                bool pushed = queue.push(
                    std::move(event),
                    stop_token
                );

                auto push_end = Clock::now();


                // Duration includes:
                //
                // mutex acquisition
                // queue insertion
                // possible waiting due to full queue
                auto push_duration =
                    push_end - push_start;


                // If queue was closed or stop was requested,
                // stop producing.
                if (!pushed) {
                    break;
                }


                ++metrics.produced_events;


                // For this baseline:
                //
                // push duration ≈ producer blocked time
                metrics.producer_blocked_time +=
                    std::chrono::duration_cast<
                        std::chrono::nanoseconds
                    >(push_duration);


                // Inspect current queue occupancy.
                std::size_t current_size =
                    queue.size();


                // Update the maximum observed occupancy.
                metrics.max_queue_occupancy =
                    std::max(
                        metrics.max_queue_occupancy,
                        current_size
                    );
            }


            // No more events will be produced.
            queue.close();
        }
    );


    // ============================================================
    // Consumer thread
    //
    // queue.pop()
    //      ↓
    // simulate slow processing
    //      ↓
    // consumed_events++
    // ============================================================

    std::jthread consumer(
        [&](std::stop_token stop_token) {

            while (true) {

                auto event =
                    queue.pop(stop_token);


                // queue closed and completely drained,
                // or stop requested.
                if (!event.has_value()) {
                    break;
                }


                // Simulate a slow downstream stage.
                std::this_thread::sleep_for(
                    consumer_delay
                );


                ++metrics.consumed_events;
            }
        }
    );


    // ============================================================
    // Wait for worker threads
    // ============================================================

    producer.join();

    consumer.join();


    // ============================================================
    // Experiment end time
    // ============================================================

    auto experiment_end = Clock::now();

    metrics.total_runtime =
        std::chrono::duration_cast<
            std::chrono::nanoseconds
        >(
            experiment_end -
            experiment_start
        );


    // ============================================================
    // Derived metrics
    // ============================================================

    double runtime_seconds =
        std::chrono::duration<double>(
            metrics.total_runtime
        ).count();


    double blocked_seconds =
        std::chrono::duration<double>(
            metrics.producer_blocked_time
        ).count();


    double producer_throughput =
        static_cast<double>(
            metrics.produced_events
        )
        / runtime_seconds;


    double consumer_throughput =
        static_cast<double>(
            metrics.consumed_events
        )
        / runtime_seconds;


    double max_queue_utilization =
        100.0 *
        static_cast<double>(
            metrics.max_queue_occupancy
        )
        /
        static_cast<double>(
            queue.capacity()
        );


    double producer_blocked_percentage =
        100.0 *
        blocked_seconds
        /
        runtime_seconds;


    // ============================================================
    // Print experiment results
    // ============================================================

    std::cout
        << "\n=== Backpressure Baseline ===\n";


    // Workload metrics
    std::cout
        << "Attempted events: "
        << metrics.attempted_events
        << '\n';

    std::cout
        << "Produced events: "
        << metrics.produced_events
        << '\n';

    std::cout
        << "Consumed events: "
        << metrics.consumed_events
        << '\n';

    std::cout
        << "Dropped events: "
        << metrics.dropped_events
        << '\n';


    // Queue metrics
    std::cout
        << "\nQueue capacity: "
        << queue.capacity()
        << '\n';

    std::cout
        << "Max queue occupancy: "
        << metrics.max_queue_occupancy
        << '\n';

    std::cout
        << "Max queue utilization: "
        << max_queue_utilization
        << " %\n";


    // Timing metrics
    std::cout
        << "\nTotal runtime: "
        << runtime_seconds
        << " s\n";

    std::cout
        << "Producer push/block time: "
        << blocked_seconds
        << " s\n";

    std::cout
        << "Producer blocked percentage: "
        << producer_blocked_percentage
        << " %\n";


    // Throughput metrics
    std::cout
        << "\nProducer throughput: "
        << producer_throughput
        << " events/s\n";

    std::cout
        << "Consumer throughput: "
        << consumer_throughput
        << " events/s\n";


    return 0;
}

Writing distributed-profiler/benchmarks/backpressure_baseline.cpp


In [7]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/benchmarks/backpressure_baseline.cpp \
    -pthread \
    -o backpressure_baseline

In [8]:
!./backpressure_baseline


=== Backpressure Baseline ===
Attempted events: 100
Produced events: 100
Consumed events: 100
Dropped events: 0

Queue capacity: 8
Max queue occupancy: 8
Max queue utilization: 100 %

Total runtime: 1.01765 s
Producer push/block time: 0.925578 s
Producer blocked percentage: 90.9525 %

Producer throughput: 98.2656 events/s
Consumer throughput: 98.2656 events/s


## Precise Producer Blocking Metrics

### Goal

The previous baseline experiment showed clear backpressure:

```text
Queue capacity reached 100%

Producer spent most of the runtime inside queue.push()

Producer throughput was forced down to consumer throughput
```

However, the previous measurement used:

```text
push duration
≈
producer blocked time
```

This is only an approximation.

The measured `push()` duration includes:

```text
mutex acquisition
condition-variable waiting
queue insertion
small function overhead
```

What we really want to measure is:

> How many times did the producer actually have to wait because the queue was full, and how long did it really wait?

---

## Why More Precise Instrumentation Is Needed

In the baseline implementation, we measured:

```cpp
auto push_start = Clock::now();

queue.push(...);

auto push_end = Clock::now();
```

Then:

```text
push_duration = push_end - push_start
```

If the queue is not full, `push_duration` may still be non-zero because `push()` performs:

```text
mutex lock
predicate check
queue insertion
mutex unlock
notification
```

Therefore:

```text
push_duration
```

is not exactly the same as:

```text
time blocked by backpressure
```

For more accurate analysis, the queue itself should detect when it actually has to wait.

---

## Desired Information from `push()`

For every push operation, we want to know:

```text
Was the push successful?

Did the producer encounter a full queue?

How long did it wait for space?
```

A useful result structure is:

```cpp
struct PushResult {
    bool success = false;
    bool blocked = false;

    std::chrono::nanoseconds
        wait_time{0};
};
```

Then:

```text
success
```

tells us whether the event entered the queue.

```text
blocked
```

tells us whether the queue was full when the producer attempted to push.

```text
wait_time
```

tells us how long the producer had to wait for free capacity.

---

## Updated Push Flow

The new `push()` behavior becomes:

```text
producer calls push()
        ↓
lock mutex
        ↓
is queue full?
   ┌────┴────┐
   │         │
  no        yes
   │         │
   │      blocked = true
   │         ↓
   │      start timer
   │         ↓
   │      wait()
   │         ↓
   │      space available
   │         ↓
   │      stop timer
   │         ↓
   │      calculate wait_time
   │
   └─────────┬─────────
             ↓
       push item
             ↓
       return PushResult
```

---

## Detecting Whether the Producer Blocks

Before calling `wait()`, the queue can inspect:

```cpp
queue_.size() >= capacity_
```

If this is true, then the producer has encountered backpressure.

Conceptually:

```cpp
bool blocked =
    queue_.size() >= capacity_;
```

If:

```text
blocked == false
```

then the item can normally be inserted without waiting for capacity.

If:

```text
blocked == true
```

then the producer must wait until the consumer removes an item.

---

## Measuring Actual Wait Time

If the queue is full:

```text
queue full
    ↓
start wait timer
    ↓
condition-variable wait
    ↓
consumer pops event
    ↓
producer wakes
    ↓
stop wait timer
```

Conceptually:

```cpp
auto wait_start = Clock::now();

not_full_.wait(...);

auto wait_end = Clock::now();
```

Then:

```cpp
wait_time =
    wait_end - wait_start;
```

This is much closer to the actual backpressure delay than timing the entire `push()` call externally.

---

## New Producer Metrics

With `PushResult`, the producer can update the metrics directly:

```text
attempt push
    ↓
PushResult
    ↓
success?
    ↓ yes
produced_events++
```

and:

```text
blocked?
    ↓ yes
producer_block_count++
producer_blocked_time += wait_time
```

This gives us two useful backpressure metrics:

```text
producer_block_count
producer_blocked_time
```

---

## Producer Block Count

`producer_block_count` tells us:

> How many push operations encountered a full queue?

For example:

```text
Total pushes:
100

Blocked pushes:
91
```

This means most pushes occurred while the pipeline was already under backpressure.

---

## Producer Blocked Time

`producer_blocked_time` tells us:

> How much total time did the producer spend waiting for queue capacity?

For example:

```text
Total runtime:
1.0 s

Producer blocked time:
0.90 s
```

This indicates that the producer spent almost the entire experiment waiting for the consumer.

---

## Average Blocking Time

Once we have:

```text
producer_block_count
producer_blocked_time
```

we can calculate:

```text
average_block_time
=
producer_blocked_time
---------------------
producer_block_count
```

For example:

```text
blocked time = 900 ms
block count = 90

average block time = 10 ms
```

If the consumer processes approximately one event every:

```text
10 ms
```

then this result would make sense.

---

## Relationship to Consumer Delay

Suppose:

```text
consumer delay = 10 ms/event
```

and the queue remains full.

Then a blocked producer usually needs to wait until:

```text
consumer finishes one event
        ↓
consumer pops another event
        ↓
one queue slot becomes available
```

Therefore, once the system reaches steady-state backpressure, the producer's waiting time per blocked push may be close to the consumer's processing interval.

Conceptually:

```text
Consumer frees ~1 slot every 10 ms

Producer waits for one slot

Average producer wait
≈ 10 ms
```

---

## Updated Metrics Structure

The relevant metrics now become:

```text
PipelineMetrics
│
├── attempted_events
├── produced_events
├── consumed_events
├── dropped_events
│
├── producer_block_count
├── producer_blocked_time
│
├── max_queue_occupancy
│
└── total_runtime
```

From these values, we can derive:

```text
producer throughput

consumer throughput

blocked percentage

average block time

queue utilization
```

---

## Derived Backpressure Metrics

### Blocked Percentage

```text
blocked_percentage
=
producer_blocked_time
---------------------
total_runtime
× 100%
```

This tells us what fraction of the experiment the producer spent waiting.

---

### Average Block Time

```text
average_block_time
=
producer_blocked_time
---------------------
producer_block_count
```

This tells us how expensive each backpressure event is on average.

---

### Blocked Push Ratio

We can also calculate:

```text
blocked_push_ratio
=
producer_block_count
--------------------
produced_events
× 100%
```

For example:

```text
produced events = 100
blocked pushes = 91

blocked push ratio = 91%
```

This shows how frequently event production encounters a full queue.

---

## Why This Matters for Profiling

For a profiling system, producer blocking may directly affect the application being measured.

Conceptually:

```text
Application thread
       ↓
record profiling event
       ↓
queue full
       ↓
wait
       ↓
application delayed
```

Therefore, precise blocking metrics help answer:

> How much overhead is the profiling infrastructure adding to the application?

This is more useful than only measuring total throughput.

---

## Expected Result

For the current workload:

```text
Total events:
100

Queue capacity:
8

Consumer delay:
10 ms
```

we expect:

```text
max queue occupancy:
8

queue utilization:
100%

producer block count:
high

producer blocked time:
high

average blocked time:
close to consumer processing interval
```

The exact numbers depend on thread scheduling, but the overall pattern should be clear.

---

## Why We Do This Before the Queue Capacity Benchmark

The next major experiment will compare:

```text
capacity = 1
capacity = 8
capacity = 64
capacity = 256
capacity = 1024
```

To compare those configurations meaningfully, we want accurate measurements of:

```text
how often the producer blocks

how long the producer blocks
```

Therefore, precise queue instrumentation should be added before the capacity experiment.

---

## Questions to Answer After This Section

After completing this section, the following questions should be answerable:

- Why is timing the entire `push()` only an approximation of blocking time?
- How can the queue determine whether a push actually encounters backpressure?
- Why should blocking time be measured inside the queue?
- What does `producer_block_count` represent?
- What does `producer_blocked_time` represent?
- How can average block time be calculated?
- Why might average block time be close to the consumer processing delay?
- Why are these metrics important for profiling overhead?
- Why should precise instrumentation be added before comparing queue capacities?

In [9]:
%%writefile distributed-profiler/include/bounded_queue.hpp

#pragma once

#include <chrono>
#include <condition_variable>
#include <cstddef>
#include <deque>
#include <mutex>
#include <optional>
#include <stop_token>
#include <utility>


// ============================================================
// Result returned by push().
//
// success:
//     Was the item successfully inserted?
//
// blocked:
//     Was the queue full when push() arrived?
//
// wait_time:
//     How long did the producer wait for queue capacity?
// ============================================================

struct PushResult {
    bool success = false;
    bool blocked = false;

    std::chrono::nanoseconds wait_time{0};
};


template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity)
        : capacity_(capacity),
          closed_(false) {}


    // ============================================================
    // Push one item into the queue.
    //
    // In addition to inserting the item, this function now
    // measures actual backpressure waiting time.
    // ============================================================

    PushResult push(
        T item,
        std::stop_token stop_token
    ) {
        using Clock = std::chrono::steady_clock;

        std::unique_lock<std::mutex> lock(mutex_);

        PushResult result;


        // --------------------------------------------------------
        // Check whether the queue is already full.
        //
        // If it is full at this point, this push operation
        // encounters backpressure.
        // --------------------------------------------------------

        result.blocked =
            queue_.size() >= capacity_;


        // --------------------------------------------------------
        // Start timing only if the producer actually needs to wait.
        // --------------------------------------------------------

        Clock::time_point wait_start;

        if (result.blocked) {
            wait_start = Clock::now();
        }


        // --------------------------------------------------------
        // Wait until:
        //
        // - queue has free capacity
        // - queue is closed
        // - stop is requested
        //
        // condition_variable_any releases mutex_ while sleeping
        // and reacquires it before returning.
        // --------------------------------------------------------

        bool ready = not_full_.wait(
            lock,
            stop_token,
            [this]() {
                return queue_.size() < capacity_
                    || closed_;
            }
        );


        // --------------------------------------------------------
        // If we actually blocked, stop the wait timer now.
        // --------------------------------------------------------

        if (result.blocked) {
            auto wait_end = Clock::now();

            result.wait_time =
                std::chrono::duration_cast<
                    std::chrono::nanoseconds
                >(
                    wait_end - wait_start
                );
        }


        // --------------------------------------------------------
        // Push cannot continue if:
        //
        // - wait was interrupted
        // - stop requested
        // - queue closed
        // --------------------------------------------------------

        if (
            !ready ||
            stop_token.stop_requested() ||
            closed_
        ) {
            result.success = false;
            return result;
        }


        // --------------------------------------------------------
        // Insert item into queue.
        // --------------------------------------------------------

        queue_.push_back(std::move(item));

        result.success = true;


        // Release mutex before notification.
        lock.unlock();

        not_empty_.notify_one();

        return result;
    }


    // ============================================================
    // Pop one item.
    // ============================================================

    std::optional<T> pop(
        std::stop_token stop_token
    ) {
        std::unique_lock<std::mutex> lock(mutex_);

        bool ready = not_empty_.wait(
            lock,
            stop_token,
            [this]() {
                return !queue_.empty()
                    || closed_;
            }
        );

        if (
            !ready ||
            stop_token.stop_requested()
        ) {
            return std::nullopt;
        }

        if (
            queue_.empty() &&
            closed_
        ) {
            return std::nullopt;
        }

        T item =
            std::move(queue_.front());

        queue_.pop_front();

        lock.unlock();

        not_full_.notify_one();

        return item;
    }


    // ============================================================
    // Close queue.
    // ============================================================

    void close() {
        {
            std::lock_guard<std::mutex> lock(mutex_);

            closed_ = true;
        }

        not_empty_.notify_all();
        not_full_.notify_all();
    }


    // ============================================================
    // Current queue occupancy.
    // ============================================================

    std::size_t size() const {
        std::lock_guard<std::mutex> lock(mutex_);

        return queue_.size();
    }


    // ============================================================
    // Maximum queue capacity.
    // ============================================================

    std::size_t capacity() const {
        return capacity_;
    }


private:
    std::deque<T> queue_;

    std::size_t capacity_;

    mutable std::mutex mutex_;

    std::condition_variable_any not_empty_;

    std::condition_variable_any not_full_;

    bool closed_;
};

Overwriting distributed-profiler/include/bounded_queue.hpp


In [10]:
%%writefile distributed-profiler/benchmarks/backpressure_baseline.cpp

#include "bounded_queue.hpp"
#include "pipeline_metrics.hpp"
#include "profile_event.hpp"

#include <algorithm>
#include <chrono>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <stop_token>
#include <thread>


int main() {

    using Clock = std::chrono::steady_clock;


    // ============================================================
    // Experiment configuration
    // ============================================================

    const std::size_t total_events = 100;

    const std::size_t queue_capacity = 8;

    const auto consumer_delay =
        std::chrono::milliseconds(10);


    // ============================================================
    // Queue and metrics
    // ============================================================

    BoundedQueue<ProfileEvent> queue(
        queue_capacity
    );

    PipelineMetrics metrics;


    // ============================================================
    // Start experiment timer.
    // ============================================================

    auto experiment_start =
        Clock::now();


    // ============================================================
    // Producer
    // ============================================================

    std::jthread producer(
        [&](std::stop_token stop_token) {

            for (
                std::size_t i = 0;
                i < total_events;
                ++i
            ) {

                ++metrics.attempted_events;


                ProfileEvent event =
                    KernelEvent{
                        static_cast<std::uint64_t>(i),
                        100,
                        0,
                        "synthetic_kernel"
                    };


                // ------------------------------------------------
                // push() now returns detailed backpressure info.
                // ------------------------------------------------

                PushResult result =
                    queue.push(
                        std::move(event),
                        stop_token
                    );


                // ------------------------------------------------
                // Stop if insertion failed.
                // ------------------------------------------------

                if (!result.success) {
                    break;
                }


                ++metrics.produced_events;


                // ------------------------------------------------
                // Record precise blocking metrics.
                //
                // Only count this push if it actually encountered
                // a full queue.
                // ------------------------------------------------

                if (result.blocked) {

                    ++metrics.producer_block_count;

                    metrics.producer_blocked_time +=
                        result.wait_time;
                }


                // ------------------------------------------------
                // Observe queue occupancy.
                // ------------------------------------------------

                std::size_t current_size =
                    queue.size();

                metrics.max_queue_occupancy =
                    std::max(
                        metrics.max_queue_occupancy,
                        current_size
                    );
            }


            // Signal that no more events will arrive.
            queue.close();
        }
    );


    // ============================================================
    // Consumer
    // ============================================================

    std::jthread consumer(
        [&](std::stop_token stop_token) {

            while (true) {

                auto event =
                    queue.pop(stop_token);


                if (!event.has_value()) {
                    break;
                }


                // Simulate slow downstream processing.
                std::this_thread::sleep_for(
                    consumer_delay
                );


                ++metrics.consumed_events;
            }
        }
    );


    // ============================================================
    // Wait for both threads.
    // ============================================================

    producer.join();

    consumer.join();


    // ============================================================
    // Stop experiment timer.
    // ============================================================

    auto experiment_end =
        Clock::now();


    metrics.total_runtime =
        std::chrono::duration_cast<
            std::chrono::nanoseconds
        >(
            experiment_end -
            experiment_start
        );


    // ============================================================
    // Convert timing values.
    // ============================================================

    double runtime_seconds =
        std::chrono::duration<double>(
            metrics.total_runtime
        ).count();


    double blocked_seconds =
        std::chrono::duration<double>(
            metrics.producer_blocked_time
        ).count();


    // ============================================================
    // Throughput
    // ============================================================

    double producer_throughput =
        static_cast<double>(
            metrics.produced_events
        )
        / runtime_seconds;


    double consumer_throughput =
        static_cast<double>(
            metrics.consumed_events
        )
        / runtime_seconds;


    // ============================================================
    // Queue utilization
    // ============================================================

    double max_queue_utilization =
        100.0 *
        static_cast<double>(
            metrics.max_queue_occupancy
        )
        /
        static_cast<double>(
            queue.capacity()
        );


    // ============================================================
    // Blocking percentage
    // ============================================================

    double producer_blocked_percentage =
        100.0 *
        blocked_seconds
        /
        runtime_seconds;


    // ============================================================
    // Average block time
    // ============================================================

    double average_block_ms = 0.0;

    if (metrics.producer_block_count > 0) {

        average_block_ms =
            std::chrono::duration<double, std::milli>(
                metrics.producer_blocked_time
            ).count()
            /
            static_cast<double>(
                metrics.producer_block_count
            );
    }


    // ============================================================
    // Blocked push ratio
    // ============================================================

    double blocked_push_ratio = 0.0;

    if (metrics.produced_events > 0) {

        blocked_push_ratio =
            100.0 *
            static_cast<double>(
                metrics.producer_block_count
            )
            /
            static_cast<double>(
                metrics.produced_events
            );
    }


    // ============================================================
    // Print results
    // ============================================================

    std::cout
        << "\n=== Precise Backpressure Metrics ===\n";


    // Workload
    std::cout
        << "Attempted events: "
        << metrics.attempted_events
        << '\n';

    std::cout
        << "Produced events: "
        << metrics.produced_events
        << '\n';

    std::cout
        << "Consumed events: "
        << metrics.consumed_events
        << '\n';

    std::cout
        << "Dropped events: "
        << metrics.dropped_events
        << '\n';


    // Queue
    std::cout
        << "\nQueue capacity: "
        << queue.capacity()
        << '\n';

    std::cout
        << "Max queue occupancy: "
        << metrics.max_queue_occupancy
        << '\n';

    std::cout
        << "Max queue utilization: "
        << max_queue_utilization
        << " %\n";


    // Backpressure
    std::cout
        << "\nProducer block count: "
        << metrics.producer_block_count
        << '\n';

    std::cout
        << "Producer blocked time: "
        << blocked_seconds
        << " s\n";

    std::cout
        << "Producer blocked percentage: "
        << producer_blocked_percentage
        << " %\n";

    std::cout
        << "Average block time: "
        << average_block_ms
        << " ms\n";

    std::cout
        << "Blocked push ratio: "
        << blocked_push_ratio
        << " %\n";


    // Throughput
    std::cout
        << "\nTotal runtime: "
        << runtime_seconds
        << " s\n";

    std::cout
        << "Producer throughput: "
        << producer_throughput
        << " events/s\n";

    std::cout
        << "Consumer throughput: "
        << consumer_throughput
        << " events/s\n";


    return 0;
}

Overwriting distributed-profiler/benchmarks/backpressure_baseline.cpp


In [11]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/benchmarks/backpressure_baseline.cpp \
    -pthread \
    -o backpressure_baseline

In [12]:
!./backpressure_baseline


=== Precise Backpressure Metrics ===
Attempted events: 100
Produced events: 100
Consumed events: 100
Dropped events: 0

Queue capacity: 8
Max queue occupancy: 8
Max queue utilization: 100 %

Producer block count: 92
Producer blocked time: 0.924606 s
Producer blocked percentage: 90.9964 %
Average block time: 10.0501 ms
Blocked push ratio: 92 %

Total runtime: 1.01609 s
Producer throughput: 98.4164 events/s
Consumer throughput: 98.4164 events/s


## Queue Capacity Experiment

### Goal

The goal of this section is to study how the capacity of a bounded queue affects backpressure.

We will keep the workload unchanged and vary only:

```text
queue capacity
```

This allows us to isolate the effect of buffering.

The central question is:

> If the producer is much faster than the consumer, how does a larger queue change producer blocking and pipeline behavior?

---

## Fixed Workload

We will keep the following parameters constant:

```text
Total events:
100

Producer delay:
0 ms

Consumer delay:
10 ms per event
```

The consumer therefore remains the bottleneck.

Its approximate processing rate is:

```text
1 event / 10 ms
≈
100 events / second
```

The producer generates events much faster than that.

---

## Queue Capacities

We will compare several queue capacities:

```text
1
8
16
32
64
128
```

Each experiment uses exactly the same producer and consumer workload.

Only the amount of buffering changes.

---

## Experimental Structure

Instead of writing a separate program for every queue size, we will refactor the baseline experiment into a reusable function.

Conceptually:

```text
run_experiment(queue_capacity)
        ↓
create queue
        ↓
create producer
        ↓
create consumer
        ↓
run workload
        ↓
collect metrics
        ↓
return result
```

Then:

```text
capacity = 1
    ↓
run experiment

capacity = 8
    ↓
run experiment

capacity = 16
    ↓
run experiment

...
```

Finally, we compare the results in one table.

---

## Why Refactor Into `run_experiment()`

The previous benchmark placed the entire experiment inside:

```cpp
main()
```

That worked for one configuration.

Now we want to repeat the same workload many times.

A reusable function makes this easier:

```cpp
ExperimentResult run_experiment(
    std::size_t queue_capacity
);
```

Conceptually:

```text
input:
queue capacity

output:
metrics for that experiment
```

This also prepares the benchmark for later experiments involving:

```text
batch size
consumer delay
overload policy
```

---

## Expected Behavior with a Small Queue

Consider:

```text
capacity = 1
```

The queue can store only one event.

The sequence will look roughly like:

```text
Producer pushes event
        ↓
queue full
        ↓
producer tries next event
        ↓
producer blocks
        ↓
consumer removes event
        ↓
producer continues
```

The producer will therefore encounter backpressure very frequently.

We expect:

```text
high block count
high blocked push ratio
high blocked time
```

---

## Expected Behavior with Capacity 8

This is similar to the baseline experiment:

```text
capacity = 8
```

The producer can initially push several events quickly:

```text
[1][2][3][4][5][6][7][8]
```

After the queue becomes full:

```text
producer blocks
```

The consumer then effectively controls how quickly new events can enter the queue.

We already observed approximately:

```text
92 blocked pushes out of 100
```

because:

```text
100 total events
-
8 initially buffered events
≈
92 blocked events
```

---

## Expected Behavior with a Larger Queue

Consider:

```text
capacity = 64
```

The producer can initially buffer many more events:

```text
Producer
   ↓
[1][2][3] ... [64]
```

Only after the queue fills does backpressure reach the producer.

Therefore, compared with a queue of capacity 8, we expect:

```text
fewer blocked pushes
less total producer blocked time
producer finishes earlier
```

However, the consumer still requires approximately:

```text
10 ms per event
```

so the total time required to consume all 100 events remains close to:

```text
1 second
```

---

## Capacity Larger Than the Entire Workload

Consider:

```text
capacity = 128
```

but:

```text
total events = 100
```

The entire workload can fit inside the queue.

Conceptually:

```text
Producer
    ↓
push all 100 events rapidly
    ↓
finish
    ↓
queue contains many pending events
    ↓
Consumer slowly drains them
```

The producer may experience almost no backpressure.

However:

```text
consumer throughput does not improve
```

The total pipeline runtime is still dominated by the consumer.

---

## Important Distinction

A larger queue can improve:

```text
producer blocking
producer completion time
short-term burst absorption
```

but it does not make the slow consumer faster.

Therefore:

```text
larger queue
!=
higher downstream processing capacity
```

The queue changes buffering behavior, not the bottleneck itself.

---

## Expected Relationship

The expected trend is:

```text
Queue capacity increases
        ↓
Producer can buffer more events
        ↓
Backpressure reaches producer later
        ↓
Block count decreases
        ↓
Producer blocked time decreases
```

However:

```text
Consumer rate remains unchanged
        ↓
Long-term pipeline throughput remains similar
```

---

## Metrics to Compare

For each queue capacity, we will collect:

```text
producer_block_count

producer_blocked_time

average_block_time

blocked_push_ratio

max_queue_occupancy

max_queue_utilization

producer_throughput

consumer_throughput

total_runtime
```

---

## Expected Results Table

The benchmark will eventually produce a table similar to:

| Queue Capacity | Block Count | Blocked Push Ratio | Blocked Time | Max Occupancy | Runtime |
|---:|---:|---:|---:|---:|---:|
| 1 | ... | ... | ... | ... | ... |
| 8 | ... | ... | ... | ... | ... |
| 16 | ... | ... | ... | ... | ... |
| 32 | ... | ... | ... | ... | ... |
| 64 | ... | ... | ... | ... | ... |
| 128 | ... | ... | ... | ... | ... |

The most important trends should be easier to see across multiple runs than in a single baseline experiment.

---

## Expected Backpressure Trend

Conceptually:

```text
Capacity = 1

Producer
   ↓
[FULL]
   ↓
block almost immediately
```

```text
Capacity = 8

Producer
   ↓
[........]
   ↓
block after small burst
```

```text
Capacity = 64

Producer
   ↓
[........................................................]
   ↓
block much later
```

```text
Capacity = 128

Producer
   ↓
entire workload fits
   ↓
little or no producer blocking
```

---

## Queue Capacity Tradeoff

A larger queue reduces immediate backpressure, but it is not free.

### Small Queue

Advantages:

```text
low memory usage
small backlog
lower potential queueing delay
```

Disadvantages:

```text
frequent producer blocking
higher application interference
poor burst absorption
```

### Large Queue

Advantages:

```text
absorbs bursts
reduces short-term producer blocking
allows producer to finish sooner
```

Disadvantages:

```text
higher memory usage
larger backlog
potentially higher event latency
```

Therefore:

```text
larger queue
!=
always better
```

---

## Profiling-System Interpretation

For a profiling system, this tradeoff matters because the producer may run on or near the application thread.

A very small queue may cause:

```text
Application
    ↓
record event
    ↓
queue full
    ↓
block
    ↓
application delayed
```

A larger queue can reduce this immediate interference:

```text
Application
    ↓
record event
    ↓
buffer event
    ↓
continue
```

but now more profiling data may remain buffered for longer periods.

So queue capacity represents a tradeoff between:

```text
application interference
        vs
memory usage
        vs
event latency
```

---

## What We Expect to Learn

This experiment should demonstrate that queue capacity mainly controls:

```text
how much burst traffic can be absorbed
```

and:

```text
how quickly backpressure reaches the producer
```

It does not fundamentally change:

```text
consumer processing capability
```

If the consumer remains the bottleneck, increasing queue capacity only delays the moment when backpressure reaches the producer.

---

## Questions to Answer After This Experiment

After completing this section, the following questions should be answerable:

- How does queue capacity affect producer blocking?
- Why does a larger queue reduce the blocked push count?
- Why might total pipeline runtime remain almost unchanged?
- Why does a larger queue not make the consumer faster?
- What happens when queue capacity is larger than the entire workload?
- What is burst absorption?
- Why can a larger queue increase event latency?
- Why is queue capacity a tradeoff between memory, latency, and producer interference?
- Why should queue capacity be treated as a tuning parameter rather than simply maximized?

In [13]:
%%writefile distributed-profiler/benchmarks/queue_capacity_benchmark.cpp

#include "bounded_queue.hpp"
#include "pipeline_metrics.hpp"
#include "profile_event.hpp"

#include <algorithm>
#include <chrono>
#include <cstddef>
#include <cstdint>
#include <iomanip>
#include <iostream>
#include <stop_token>
#include <thread>
#include <vector>


struct ExperimentResult {
    std::size_t queue_capacity = 0;

    PipelineMetrics metrics;

    double runtime_seconds = 0.0;
    double producer_throughput = 0.0;
    double consumer_throughput = 0.0;

    double blocked_seconds = 0.0;
    double blocked_percentage = 0.0;
    double average_block_ms = 0.0;
    double blocked_push_ratio = 0.0;

    double max_queue_utilization = 0.0;
};


// ============================================================
// Run one producer-consumer experiment with a given
// queue capacity.
//
// Fixed workload:
//
// total_events   = 100
// consumer_delay = 10 ms
//
// Only queue_capacity changes.
// ============================================================

ExperimentResult run_experiment(
    std::size_t queue_capacity,
    std::size_t total_events,
    std::chrono::milliseconds consumer_delay
) {
    using Clock = std::chrono::steady_clock;

    BoundedQueue<ProfileEvent> queue(
        queue_capacity
    );

    PipelineMetrics metrics;


    // ============================================================
    // Start experiment timer.
    // ============================================================

    auto experiment_start =
        Clock::now();


    // ============================================================
    // Producer
    //
    // Producer generates events as fast as possible.
    //
    // If the queue becomes full:
    //
    // producer
    //    ↓
    // push()
    //    ↓
    // wait for free queue capacity
    //
    // PushResult tells us:
    //
    // success
    // blocked
    // wait_time
    // ============================================================

    std::jthread producer(
        [&](std::stop_token stop_token) {

            for (
                std::size_t i = 0;
                i < total_events;
                ++i
            ) {
                ++metrics.attempted_events;


                ProfileEvent event =
                    KernelEvent{
                        static_cast<std::uint64_t>(i),
                        100,
                        0,
                        "synthetic_kernel"
                    };


                PushResult result =
                    queue.push(
                        std::move(event),
                        stop_token
                    );


                if (!result.success) {
                    break;
                }


                ++metrics.produced_events;


                // ------------------------------------------------
                // Record precise backpressure metrics.
                // ------------------------------------------------

                if (result.blocked) {

                    ++metrics.producer_block_count;

                    metrics.producer_blocked_time +=
                        result.wait_time;
                }


                // ------------------------------------------------
                // Observe queue occupancy.
                //
                // queue.size() gives a thread-safe snapshot.
                // ------------------------------------------------

                std::size_t current_size =
                    queue.size();


                metrics.max_queue_occupancy =
                    std::max(
                        metrics.max_queue_occupancy,
                        current_size
                    );
            }


            // No more events will be produced.
            queue.close();
        }
    );


    // ============================================================
    // Consumer
    //
    // Consumer intentionally processes events slowly.
    //
    // queue.pop()
    //      ↓
    // sleep 10 ms
    //      ↓
    // consumed_events++
    //
    // This makes the consumer the bottleneck.
    // ============================================================

    std::jthread consumer(
        [&](std::stop_token stop_token) {

            while (true) {

                auto event =
                    queue.pop(stop_token);


                if (!event.has_value()) {
                    break;
                }


                std::this_thread::sleep_for(
                    consumer_delay
                );


                ++metrics.consumed_events;
            }
        }
    );


    // ============================================================
    // Wait for both threads to finish.
    // ============================================================

    producer.join();

    consumer.join();


    // ============================================================
    // Stop experiment timer.
    // ============================================================

    auto experiment_end =
        Clock::now();


    metrics.total_runtime =
        std::chrono::duration_cast<
            std::chrono::nanoseconds
        >(
            experiment_end -
            experiment_start
        );


    // ============================================================
    // Derived metrics
    // ============================================================

    double runtime_seconds =
        std::chrono::duration<double>(
            metrics.total_runtime
        ).count();


    double blocked_seconds =
        std::chrono::duration<double>(
            metrics.producer_blocked_time
        ).count();


    double producer_throughput =
        static_cast<double>(
            metrics.produced_events
        )
        / runtime_seconds;


    double consumer_throughput =
        static_cast<double>(
            metrics.consumed_events
        )
        / runtime_seconds;


    double blocked_percentage =
        100.0 *
        blocked_seconds
        /
        runtime_seconds;


    double max_queue_utilization =
        100.0 *
        static_cast<double>(
            metrics.max_queue_occupancy
        )
        /
        static_cast<double>(
            queue_capacity
        );


    double average_block_ms = 0.0;

    if (
        metrics.producer_block_count > 0
    ) {
        average_block_ms =
            std::chrono::duration<
                double,
                std::milli
            >(
                metrics.producer_blocked_time
            ).count()
            /
            static_cast<double>(
                metrics.producer_block_count
            );
    }


    double blocked_push_ratio = 0.0;

    if (
        metrics.produced_events > 0
    ) {
        blocked_push_ratio =
            100.0 *
            static_cast<double>(
                metrics.producer_block_count
            )
            /
            static_cast<double>(
                metrics.produced_events
            );
    }


    // ============================================================
    // Package everything into ExperimentResult.
    // ============================================================

    ExperimentResult result;

    result.queue_capacity =
        queue_capacity;

    result.metrics =
        metrics;

    result.runtime_seconds =
        runtime_seconds;

    result.producer_throughput =
        producer_throughput;

    result.consumer_throughput =
        consumer_throughput;

    result.blocked_seconds =
        blocked_seconds;

    result.blocked_percentage =
        blocked_percentage;

    result.average_block_ms =
        average_block_ms;

    result.blocked_push_ratio =
        blocked_push_ratio;

    result.max_queue_utilization =
        max_queue_utilization;


    return result;
}


int main() {

    // ============================================================
    // Fixed workload.
    // ============================================================

    const std::size_t total_events =
        100;

    const auto consumer_delay =
        std::chrono::milliseconds(10);


    // ============================================================
    // Queue capacities to compare.
    // ============================================================

    const std::vector<std::size_t> capacities = {
        1,
        8,
        16,
        32,
        64,
        128
    };


    std::vector<ExperimentResult> results;

    results.reserve(
        capacities.size()
    );


    // ============================================================
    // Run exactly the same workload for every queue capacity.
    //
    // Queue capacity is the only variable.
    // ============================================================

    for (
        std::size_t capacity :
        capacities
    ) {

        ExperimentResult result =
            run_experiment(
                capacity,
                total_events,
                consumer_delay
            );


        results.push_back(
            std::move(result)
        );
    }


    // ============================================================
    // Print comparison table.
    // ============================================================

    std::cout
        << "\n=== Queue Capacity Experiment ===\n\n";


    std::cout
        << std::left
        << std::setw(10)
        << "Capacity"

        << std::setw(12)
        << "Blocks"

        << std::setw(14)
        << "Block Ratio"

        << std::setw(14)
        << "Block Time"

        << std::setw(14)
        << "Avg Block"

        << std::setw(12)
        << "Max Queue"

        << std::setw(14)
        << "Queue Util"

        << std::setw(12)
        << "Runtime"

        << '\n';


    std::cout
        << std::string(102, '-')
        << '\n';


    for (
        const auto& result :
        results
    ) {

        std::cout
            << std::left

            << std::setw(10)
            << result.queue_capacity

            << std::setw(12)
            << result.metrics.producer_block_count

            << std::setw(14)
            << (
                std::to_string(
                    result.blocked_push_ratio
                ).substr(0, 5)
                + "%"
            )

            << std::setw(14)
            << result.blocked_seconds

            << std::setw(14)
            << result.average_block_ms

            << std::setw(12)
            << result.metrics.max_queue_occupancy

            << std::setw(14)
            << (
                std::to_string(
                    result.max_queue_utilization
                ).substr(0, 6)
                + "%"
            )

            << std::setw(12)
            << result.runtime_seconds

            << '\n';
    }


    // ============================================================
    // Print throughput separately.
    //
    // We expect throughput to remain similar because the
    // consumer is still the bottleneck.
    // ============================================================

    std::cout
        << "\nThroughput comparison\n\n";


    std::cout
        << std::left
        << std::setw(12)
        << "Capacity"

        << std::setw(22)
        << "Producer events/s"

        << std::setw(22)
        << "Consumer events/s"

        << '\n';


    std::cout
        << std::string(56, '-')
        << '\n';


    for (
        const auto& result :
        results
    ) {

        std::cout
            << std::left

            << std::setw(12)
            << result.queue_capacity

            << std::setw(22)
            << result.producer_throughput

            << std::setw(22)
            << result.consumer_throughput

            << '\n';
    }


    return 0;
}

Writing distributed-profiler/benchmarks/queue_capacity_benchmark.cpp


In [14]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/benchmarks/queue_capacity_benchmark.cpp \
    -pthread \
    -o queue_capacity_benchmark

In [15]:
!./queue_capacity_benchmark


=== Queue Capacity Experiment ===

Capacity  Blocks      Block Ratio   Block Time    Avg Block     Max Queue   Queue Util    Runtime     
------------------------------------------------------------------------------------------------------
1         99          99.00%        0.996971      10.0704       1           100.00%       1.0182      
8         91          91.00%        0.925596      10.1714       8           100.00%       1.01793     
16        83          83.00%        0.844216      10.1713       16          100.00%       1.01785     
32        67          67.00%        0.681501      10.1717       32          100.00%       1.01765     
64        35          35.00%        0.355781      10.1652       64          100.00%       1.01618     
128       0           0.000%        0             0             99          77.343%       1.01642     

Throughput comparison

Capacity    Producer events/s     Consumer events/s     
--------------------------------------------------------
1 

## Overload Policy Comparison

### Goal

The previous experiments used a bounded queue with one overload behavior:

```text
Queue full
    ↓
Block producer
    ↓
Wait until consumer frees space
```

This guarantees that no profiling events are lost.

However, for a real profiling system, blocking the producer may interfere with the application being measured.

The goal of this section is to compare three overload policies:

```text
Block Producer
Drop Newest
Drop Oldest
```

and study the tradeoff between:

```text
data completeness
application interference
freshness of profiling data
```

---

## Why Overload Policy Matters

Consider a profiling pipeline:

```text
Application / GPU Runtime
        ↓
Profiling Event Producer
        ↓
Bounded Queue
        ↓
Slow Consumer / Network / Storage
```

If the consumer cannot keep up, the queue eventually becomes full.

At that point, the system must make a decision.

It can:

```text
wait
```

or:

```text
drop data
```

There is no perfect policy.

Each choice optimizes a different objective.

---

## Fixed Workload

We will use the same overloaded workload for all policies:

```text
Total events:
100

Queue capacity:
8

Producer delay:
0 ms

Consumer delay:
10 ms per event
```

The producer generates events much faster than the consumer can process them.

Therefore:

```text
producer rate
>>
consumer rate
```

and overload is guaranteed.

Only the queue policy will change.

---

# Policy 1 — Block Producer

This is the behavior we already implemented.

When the queue is full:

```text
Producer
   ↓
push(event)
   ↓
Queue full
   ↓
wait
   ↓
Consumer removes event
   ↓
space becomes available
   ↓
Producer continues
```

Example:

```text
Queue capacity = 4

[A][B][C][D]

New event E arrives

Producer waits

Consumer removes A

[B][C][D][ ]

Producer inserts E

[B][C][D][E]
```

No event is lost.

---

## Advantages of Blocking

```text
No data loss

Preserves complete event stream

Simple semantics

Good for exact tracing
```

If every profiling event is required for correctness, blocking may be necessary.

For example:

```text
complete execution trace
precise event reconstruction
debugging
```

may require all events to be preserved.

---

## Disadvantages of Blocking

The producer may be running on or near the application thread.

Therefore:

```text
Application
    ↓
profiling event
    ↓
queue full
    ↓
producer blocks
    ↓
application delayed
```

This introduces observer overhead.

For a profiler, this is dangerous because:

```text
profiling system
        ↓
changes application behavior
        ↓
measured performance becomes less accurate
```

So a lossless profiler can potentially distort the workload it is trying to measure.

---

# Policy 2 — Drop Newest

With Drop Newest, the producer never waits for queue capacity.

If the queue is full:

```text
existing queue stays unchanged
```

and the new event is discarded.

Example:

```text
Queue capacity = 4

[A][B][C][D]

New event E arrives

Queue full

E is dropped

[A][B][C][D]
```

The producer continues immediately.

---

## Advantages of Drop Newest

```text
Producer does not block

Low application interference

Existing queued events are preserved

Simple overload behavior
```

This can be useful when minimizing profiling overhead is more important than preserving every event.

---

## Disadvantages of Drop Newest

New profiling information is lost.

For example:

```text
[A][B][C][D] are waiting

E arrives
↓
E dropped

F arrives
↓
F dropped

G arrives
↓
G dropped
```

The queue may contain relatively old information while newer events continue to be discarded.

Therefore:

```text
Drop Newest
```

favors:

```text
older queued data
```

over:

```text
newer incoming data
```

---

# Policy 3 — Drop Oldest

Drop Oldest also avoids producer blocking.

However, instead of dropping the incoming event, it removes the oldest event already in the queue.

Example:

```text
Queue capacity = 4

[A][B][C][D]

New event E arrives
```

The oldest event is:

```text
A
```

So:

```text
drop A
```

and insert E:

```text
[B][C][D][E]
```

The queue remains full, but it contains more recent data.

---

## Advantages of Drop Oldest

```text
Producer does not block

Low application interference

Newest data is preserved

Useful for live monitoring
```

This policy can be useful when the profiler cares more about the current state than a complete historical trace.

For example:

```text
live telemetry
real-time dashboards
recent utilization statistics
health monitoring
```

---

## Disadvantages of Drop Oldest

The system loses events that may already have spent time waiting in the queue.

This means processing effort associated with keeping those events buffered is wasted.

It also destroys a complete chronological trace.

Therefore:

```text
Drop Oldest
```

is usually unsuitable when every event is required.

---

# High-Level Comparison

| Policy | Producer Blocking | Data Loss | Preserves Old Data | Preserves New Data |
|---|---|---|---|---|
| Block Producer | Yes | No | Yes | Yes |
| Drop Newest | No | Yes | Yes | No |
| Drop Oldest | No | Yes | No | Yes |

---

## Expected Behavior

### Block Producer

We expect:

```text
produced events ≈ 100
consumed events ≈ 100
dropped events = 0

high block count
high blocked time

runtime ≈ 1 second
```

The consumer processes all events.

---

### Drop Newest

The producer should finish very quickly because it never waits.

However, once the queue becomes full:

```text
many incoming events
will be dropped
```

We expect:

```text
producer blocking ≈ 0
dropped events > 0

consumed events << 100
```

The exact number depends on thread scheduling.

---

### Drop Oldest

The producer also avoids blocking.

When overload occurs:

```text
old queued events are replaced by newer events
```

Therefore we also expect:

```text
producer blocking ≈ 0
dropped events > 0

consumed events << 100
```

The number of consumed events may be similar to Drop Newest, but the identity of those events will be different.

---

# Why Event Identity Matters

Suppose the producer generates event IDs:

```text
0
1
2
3
...
99
```

Under Drop Newest, the consumer may observe something closer to:

```text
0
1
2
3
4
5
...
```

while later events are discarded during overload.

Under Drop Oldest, the consumer may observe:

```text
some early events
+
many events near the end of the stream
```

because old queued events are continuously replaced.

Therefore, comparing only:

```text
drop count
```

is not enough.

We should also inspect:

```text
which event IDs survived
```

This helps clearly demonstrate the semantic difference between:

```text
Drop Newest
```

and:

```text
Drop Oldest
```

---

# Metrics to Collect

For each policy, we should measure:

```text
attempted_events

produced_events

consumed_events

dropped_events

drop_rate

producer_block_count

producer_blocked_time

blocked_push_ratio

total_runtime
```

We may also record:

```text
consumed event IDs
```

to understand which events survive under each policy.

---

# Drop Rate

The drop rate can be calculated as:

```text
drop rate
=
dropped events
--------------
attempted events
× 100%
```

This directly measures how much profiling information is lost.

---

# Application Interference vs Data Loss

The central tradeoff is:

```text
Block Producer
       ↓
complete data
       ↓
higher application interference
```

versus:

```text
Drop Policies
       ↓
lower producer interference
       ↓
incomplete profiling data
```

This can be summarized as:

```text
Data completeness
        ↑
        |
Block   |
        |
--------+----------------→ Lower application overhead
        |
        | Drop policies
        |
```

There is no universally correct policy.

The correct choice depends on what the profiling system is trying to achieve.

---

# Profiling Use Cases

### Exact Trace Collection

If we need:

```text
complete kernel sequence
complete memory trace
precise event ordering
```

then dropping events may make the trace unusable.

A blocking or otherwise lossless design is more appropriate.

---

### Low-Overhead Sampling

If the profiler is collecting approximate performance information:

```text
utilization
sampling
periodic statistics
```

then dropping some events may be acceptable.

Reducing application interference may be more important.

---

### Live Telemetry

For live monitoring:

```text
recent data
```

is often more useful than:

```text
old queued data
```

Therefore Drop Oldest may be preferable.

---

# Important Profiling Insight

A profiler should not automatically assume:

```text
zero data loss
=
best design
```

Because preserving every event may require enough synchronization and waiting to significantly perturb the application.

Likewise:

```text
zero producer blocking
```

is not automatically best because the resulting profiling trace may become incomplete.

The design goal is to choose an overload policy that matches the semantics of the profiling data.

---

# Implementation Direction

We can represent the policy explicitly:

```cpp
enum class OverloadPolicy {
    Block,
    DropNewest,
    DropOldest
};
```

Then the queue behavior becomes:

```text
push(event)
    ↓
queue has space?
    │
    ├── yes
    │     ↓
    │   insert event
    │
    └── no
          ↓
      inspect policy
          │
          ├── Block
          │     ↓
          │    wait
          │
          ├── DropNewest
          │     ↓
          │    discard incoming event
          │
          └── DropOldest
                ↓
              remove front
                ↓
              insert new event
```

The result returned by `push()` should tell us what happened.

Conceptually:

```cpp
struct PushResult {
    bool success;
    bool blocked;
    bool dropped;

    std::chrono::nanoseconds wait_time;
};
```

We may later refine the result further so that we can distinguish:

```text
incoming event dropped
```

from:

```text
old queued event dropped
```

---

# Expected Experimental Result

We expect a comparison resembling:

| Policy | Block Count | Block Time | Dropped Events | Drop Rate | Runtime |
|---|---:|---:|---:|---:|---:|
| Block | High | High | 0 | 0% | ~1 s |
| Drop Newest | 0 | ~0 | High | High | Much lower |
| Drop Oldest | 0 | ~0 | High | High | Much lower |

However, runtime must be interpreted carefully.

For the drop policies, fewer events reach the consumer.

Therefore:

```text
shorter runtime
```

does not mean:

```text
consumer became faster
```

It means:

```text
less work survived overload
```

This distinction is very important when interpreting benchmark results.

---

# Key Questions to Answer After This Experiment

After completing this section, the following questions should be answerable:

- What should a bounded queue do when it becomes full?
- What is the difference between Block, Drop Newest, and Drop Oldest?
- Why does blocking preserve data but increase application interference?
- Why do drop policies reduce producer overhead?
- What type of data does Drop Newest preserve?
- What type of data does Drop Oldest preserve?
- Why is Drop Oldest useful for live telemetry?
- Why is dropping events dangerous for exact tracing?
- Why is shorter runtime under a drop policy not necessarily better performance?
- How should a profiling system choose an overload policy?
- Why is overload behavior part of API and system design rather than just an implementation detail?

In [16]:
%%writefile distributed-profiler/include/bounded_queue.hpp

#pragma once

#include <chrono>
#include <condition_variable>
#include <cstddef>
#include <deque>
#include <mutex>
#include <optional>
#include <stop_token>
#include <utility>


// ============================================================
// Queue overload behavior
// ============================================================

enum class OverloadPolicy {
    Block,
    DropNewest,
    DropOldest
};


// ============================================================
// Result returned by push()
// ============================================================

struct PushResult {
    bool success = false;

    bool blocked = false;

    bool dropped = false;

    std::chrono::nanoseconds wait_time{0};
};


template <typename T>
class BoundedQueue {
public:

    // ============================================================
    // Constructor
    // ============================================================

    explicit BoundedQueue(
        std::size_t capacity,
        OverloadPolicy policy
    )
        : capacity_(capacity),
          policy_(policy),
          closed_(false) {}


    // ============================================================
    // push()
    //
    //
    // 1. Lock mutex
    //
    // 2. Check whether queue is closed
    //
    // 3. If queue has free space:
    //      - insert item
    //      - success = true
    //
    // 4. If queue is full:
    //
    //      policy == Block
    //          - mark blocked
    //          - measure wait time
    //          - wait until space / closed / stop requested
    //          - insert item if possible
    //
    //      policy == DropNewest
    //          - discard incoming item
    //          - dropped = true
    //
    //      policy == DropOldest
    //          - remove queue_.front()
    //          - insert new item
    //          - dropped = true
    //
    // 5. Unlock before notify
    //
    // 6. Return PushResult
    // ============================================================

    PushResult push(
        T item,
        std::stop_token stop_token
    ) {
        using Clock = std::chrono::steady_clock;

        std::unique_lock<std::mutex> lock(mutex_);


        PushResult result;

        // --------------------------------------------------------
        // Queue already closed.
        // --------------------------------------------------------
        if (closed_) {
            return result;
        }

        // --------------------------------------------------------
        // Fast path:
        //
        // queue still has free capacity.
        // --------------------------------------------------------
        if (queue_.size() < capacity_) {
            queue_.push_back(
                std::move(item)
            );

            result.success = true;

            not_empty_.notify_one();

            return result;
        }


        // ========================================================
        // Queue is full.
        //
        // Select behavior based on overload policy.
        // ========================================================

        switch (policy_) {
            // ====================================================
            // Block Producer
            // ====================================================
                case OverloadPolicy::Block: {

                result.blocked = true;

                auto wait_start =
                    Clock::now();


                bool ready =
                    not_full_.wait(
                        lock,
                        stop_token,
                        [this]() {
                            return queue_.size() < capacity_
                                || closed_;
                        }
                    );


                auto wait_end =
                    Clock::now();


                result.wait_time =
                    std::chrono::duration_cast<
                        std::chrono::nanoseconds
                    >(
                        wait_end -
                        wait_start
                    );


                if (
                    !ready ||
                    stop_token.stop_requested() ||
                    closed_
                ) {
                    return result;
                }


                queue_.push_back(
                    std::move(item)
                );


                result.success = true;


                lock.unlock();

                not_empty_.notify_one();

                return result;

        }
            // ====================================================
            // Drop Newest
            //
            // Incoming event is discarded.
            // Existing queue stays unchanged.
            // ====================================================
                case OverloadPolicy::DropNewest: {

                    result.success = false;

                    result.dropped = true;

                    // No queue modification.
                    // No producer blocking.

                    return result;
                }

              // ====================================================
            // Drop Oldest
            //
            // Remove oldest queued item,
            // then insert incoming item.
            // ====================================================

            case OverloadPolicy::DropOldest: {

                queue_.pop_front();

                queue_.push_back(
                    std::move(item)
                );


                result.success = true;

                result.dropped = true;


                lock.unlock();

                not_empty_.notify_one();

                return result;
            }
        }
        return result;
    }


    // ============================================================
    // pop()
    //
    // Same behavior as previous implementation.
    // ============================================================

    std::optional<T> pop(
        std::stop_token stop_token
    ) {
        //
        // lock
        // wait until:
        //      queue not empty
        //      OR closed
        //      OR stop requested
        //
        // if no item available:
        //      return nullopt
        //
        // otherwise:
        //      move front item
        //      pop front
        //      unlock
        //      notify producer
        //      return item
        std::unique_lock<std::mutex> lock(mutex_);


        bool ready =
            not_empty_.wait(
                lock,
                stop_token,
                [this]() {
                    return !queue_.empty()
                        || closed_;
                }
            );


        if (
            !ready ||
            stop_token.stop_requested()
        ) {
            return std::nullopt;
        }


        if (
            queue_.empty() &&
            closed_
        ) {
            return std::nullopt;
        }


        T item =
            std::move(
                queue_.front()
            );


        queue_.pop_front();


        lock.unlock();

        not_full_.notify_one();


        return item;
    }


    // ============================================================
    // close()
    // ============================================================

    void close() {
        // lock
        // closed_ = true
        // unlock
        //
        // notify all waiters
        {
            std::lock_guard<std::mutex> lock(
                mutex_
            );

            closed_ = true;
        }


        not_empty_.notify_all();

        not_full_.notify_all();
    }


    // ============================================================
    // Current queue occupancy
    // ============================================================

    std::size_t size() const {

        // lock mutex
        // return queue_.size()

        std::lock_guard<std::mutex> lock(
            mutex_
        );

        return queue_.size();
    }


    // ============================================================
    // Queue capacity
    // ============================================================

    std::size_t capacity() const {
        return capacity_;
    }


    // ============================================================
    // Current overload policy
    // ============================================================

    OverloadPolicy policy() const {
        return policy_;
    }


private:

    std::deque<T> queue_;

    std::size_t capacity_;

    OverloadPolicy policy_;

    mutable std::mutex mutex_;

    std::condition_variable_any not_empty_;

    std::condition_variable_any not_full_;

    bool closed_;
};

Overwriting distributed-profiler/include/bounded_queue.hpp


In [17]:
%%writefile distributed-profiler/benchmarks/overload_policy_benchmark.cpp

#include "bounded_queue.hpp"
#include "pipeline_metrics.hpp"
#include "profile_event.hpp"

#include <chrono>
#include <cstddef>
#include <cstdint>
#include <iomanip>
#include <iostream>
#include <stop_token>
#include <string>
#include <thread>
#include <variant>
#include <vector>


// ============================================================
// Result of one policy experiment
// ============================================================

struct ExperimentResult {

    OverloadPolicy policy;

    PipelineMetrics metrics;

    double runtime_seconds = 0.0;

    double drop_rate = 0.0;

    double blocked_push_ratio = 0.0;

    double average_block_ms = 0.0;

    std::vector<std::uint64_t>
        consumed_event_ids;
};


// ============================================================
// Convert enum to readable text.
// ============================================================

std::string policy_name(
    OverloadPolicy policy
) {

    switch (policy) {

        case OverloadPolicy::Block:
            return "Block";

        case OverloadPolicy::DropNewest:
            return "DropNewest";

        case OverloadPolicy::DropOldest:
            return "DropOldest";
    }

    return "Unknown";
}


// ============================================================
// Extract event ID.
//
// In this benchmark we store the event ID inside
// KernelEvent::timestamp_ns.
// ============================================================

std::uint64_t get_event_id(
    const ProfileEvent& event
) {

    return std::visit(
        [](const auto& e) -> std::uint64_t {
            return e.timestamp_ns;
        },
        event
    );
}


// ============================================================
// Run one overload-policy experiment.
// ============================================================

ExperimentResult run_experiment(
    OverloadPolicy policy,
    std::size_t total_events,
    std::size_t queue_capacity,
    std::chrono::milliseconds consumer_delay
) {

    using Clock =
        std::chrono::steady_clock;


    BoundedQueue<ProfileEvent> queue(
        queue_capacity,
        policy
    );


    PipelineMetrics metrics;


    std::vector<std::uint64_t>
        consumed_event_ids;


    auto experiment_start =
        Clock::now();


    // ============================================================
    // Producer
    // ============================================================

    std::jthread producer(

        [&](std::stop_token stop_token) {

            for (
                std::size_t i = 0;
                i < total_events;
                ++i
            ) {

                ++metrics.attempted_events;


                ProfileEvent event =
                    KernelEvent{
                        static_cast<std::uint64_t>(i),
                        100,
                        0,
                        "synthetic_kernel"
                    };


                PushResult result =
                    queue.push(
                        std::move(event),
                        stop_token
                    );


                // ------------------------------------------------
                // Count incoming events that successfully entered
                // the queue.
                // ------------------------------------------------

                if (result.success) {
                    ++metrics.produced_events;
                }


                // ------------------------------------------------
                // Backpressure metrics.
                // ------------------------------------------------

                if (result.blocked) {

                    ++metrics.producer_block_count;

                    metrics.producer_blocked_time +=
                        result.wait_time;
                }


                // ------------------------------------------------
                // Drop metrics.
                //
                // DropNewest:
                //     incoming event was dropped.
                //
                // DropOldest:
                //     an existing queued event was dropped.
                // ------------------------------------------------

                if (result.dropped) {
                    ++metrics.dropped_events;
                }


                // ------------------------------------------------
                // Observe queue occupancy.
                // ------------------------------------------------

                std::size_t current_size =
                    queue.size();


                if (
                    current_size >
                    metrics.max_queue_occupancy
                ) {
                    metrics.max_queue_occupancy =
                        current_size;
                }
            }


            queue.close();
        }
    );


    // ============================================================
    // Consumer
    // ============================================================

    std::jthread consumer(

        [&](std::stop_token stop_token) {

            while (true) {

                auto event =
                    queue.pop(
                        stop_token
                    );


                if (!event.has_value()) {
                    break;
                }


                // ------------------------------------------------
                // Record which event survived overload.
                // ------------------------------------------------

                consumed_event_ids.push_back(
                    get_event_id(
                        *event
                    )
                );


                // ------------------------------------------------
                // Simulate slow downstream processing.
                // ------------------------------------------------

                std::this_thread::sleep_for(
                    consumer_delay
                );


                ++metrics.consumed_events;
            }
        }
    );


    // ============================================================
    // Wait for both threads.
    // ============================================================

    producer.join();

    consumer.join();


    auto experiment_end =
        Clock::now();


    metrics.total_runtime =
        std::chrono::duration_cast<
            std::chrono::nanoseconds
        >(
            experiment_end -
            experiment_start
        );


    // ============================================================
    // Derived metrics
    // ============================================================

    double runtime_seconds =
        std::chrono::duration<double>(
            metrics.total_runtime
        ).count();


    double drop_rate = 0.0;

    if (
        metrics.attempted_events > 0
    ) {

        drop_rate =
            100.0 *
            static_cast<double>(
                metrics.dropped_events
            )
            /
            static_cast<double>(
                metrics.attempted_events
            );
    }


    double blocked_push_ratio = 0.0;

    if (
        metrics.attempted_events > 0
    ) {

        blocked_push_ratio =
            100.0 *
            static_cast<double>(
                metrics.producer_block_count
            )
            /
            static_cast<double>(
                metrics.attempted_events
            );
    }


    double average_block_ms = 0.0;

    if (
        metrics.producer_block_count > 0
    ) {

        average_block_ms =
            std::chrono::duration<
                double,
                std::milli
            >(
                metrics.producer_blocked_time
            ).count()
            /
            static_cast<double>(
                metrics.producer_block_count
            );
    }


    // ============================================================
    // Build result.
    // ============================================================

    ExperimentResult result;

    result.policy =
        policy;

    result.metrics =
        metrics;

    result.runtime_seconds =
        runtime_seconds;

    result.drop_rate =
        drop_rate;

    result.blocked_push_ratio =
        blocked_push_ratio;

    result.average_block_ms =
        average_block_ms;

    result.consumed_event_ids =
        std::move(
            consumed_event_ids
        );


    return result;
}


// ============================================================
// Print event IDs that survived.
// ============================================================

void print_consumed_ids(
    const ExperimentResult& result
) {

    std::cout
        << "\nConsumed IDs for "
        << policy_name(
            result.policy
        )
        << ":\n";


    for (
        std::uint64_t id :
        result.consumed_event_ids
    ) {

        std::cout
            << id
            << ' ';
    }


    std::cout
        << '\n';
}


// ============================================================
// main
// ============================================================

int main() {

    // ============================================================
    // Fixed overloaded workload.
    // ============================================================

    const std::size_t total_events =
        100;


    const std::size_t queue_capacity =
        8;


    const auto consumer_delay =
        std::chrono::milliseconds(
            10
        );


    // ============================================================
    // Policies to compare.
    // ============================================================

    const std::vector<OverloadPolicy>
        policies = {

            OverloadPolicy::Block,

            OverloadPolicy::DropNewest,

            OverloadPolicy::DropOldest
        };


    std::vector<ExperimentResult>
        results;


    results.reserve(
        policies.size()
    );


    // ============================================================
    // Run experiments.
    // ============================================================

    for (
        OverloadPolicy policy :
        policies
    ) {

        ExperimentResult result =
            run_experiment(
                policy,
                total_events,
                queue_capacity,
                consumer_delay
            );


        results.push_back(
            std::move(result)
        );
    }


    // ============================================================
    // Print surviving event IDs.
    // ============================================================

    for (
        const auto& result :
        results
    ) {

        print_consumed_ids(
            result
        );
    }


    // ============================================================
    // Summary table.
    // ============================================================

    std::cout
        << "\n=== Overload Policy Comparison ===\n\n";


    std::cout
        << std::left

        << std::setw(15)
        << "Policy"

        << std::setw(12)
        << "Attempted"

        << std::setw(12)
        << "Produced"

        << std::setw(12)
        << "Consumed"

        << std::setw(12)
        << "Dropped"

        << std::setw(14)
        << "Drop Rate"

        << std::setw(12)
        << "Blocks"

        << std::setw(14)
        << "Block Ratio"

        << std::setw(14)
        << "Avg Block"

        << std::setw(12)
        << "Runtime"

        << '\n';


    std::cout
        << std::string(
            117,
            '-'
        )
        << '\n';


    for (
        const auto& result :
        results
    ) {

        std::cout
            << std::left

            << std::setw(15)
            << policy_name(
                result.policy
            )

            << std::setw(12)
            << result.metrics.attempted_events

            << std::setw(12)
            << result.metrics.produced_events

            << std::setw(12)
            << result.metrics.consumed_events

            << std::setw(12)
            << result.metrics.dropped_events

            << std::setw(14)
            << result.drop_rate

            << std::setw(12)
            << result.metrics.producer_block_count

            << std::setw(14)
            << result.blocked_push_ratio

            << std::setw(14)
            << result.average_block_ms

            << std::setw(12)
            << result.runtime_seconds

            << '\n';
    }


    return 0;
}

Writing distributed-profiler/benchmarks/overload_policy_benchmark.cpp


In [18]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/benchmarks/overload_policy_benchmark.cpp \
    -pthread \
    -o overload_policy_benchmark

In [19]:
!./overload_policy_benchmark


Consumed IDs for Block:
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 

Consumed IDs for DropNewest:
0 1 2 3 4 5 6 7 

Consumed IDs for DropOldest:
92 93 94 95 96 97 98 99 

=== Overload Policy Comparison ===

Policy         Attempted   Produced    Consumed    Dropped     Drop Rate     Blocks      Block Ratio   Avg Block     Runtime     
---------------------------------------------------------------------------------------------------------------------
Block          100         100         100         0           0             91          91            10.1377       1.01498     
DropNewest     100         8           8           92          92            0           0             0             0.0824372   
DropOldest     100         100       

## Batching + Backpressure

### Goal

The previous experiments studied two ways to manage overload:

```text
Queue Capacity
Overload Policy
```

Those experiments focused on what happens when the consumer is slower than the producer.

Now we want to study a different question:

> Can we make the consumer more efficient so that backpressure happens less often?

The main technique is:

```text
Batching
```

Instead of paying downstream processing overhead for every individual event, the consumer will collect multiple events and process them together.

---

## Why Batching Helps

Without batching, the consumer behaves like:

```text
pop event
    ↓
process event
    ↓
pay downstream overhead

pop event
    ↓
process event
    ↓
pay downstream overhead
```

If the downstream operation has a significant fixed cost, repeating that cost for every event is inefficient.

Examples of downstream fixed costs include:

```text
network send syscall

database transaction

serialization setup

lock acquisition

collector RPC

buffer flush
```

Batching amortizes this fixed overhead across multiple events.

---

## Non-Batched Consumer

Suppose every downstream operation has:

```text
fixed overhead = 2 ms
```

and every event also requires:

```text
per-event processing cost = 0.1 ms
```

For a single event:

```text
total cost
=
2 ms
+
0.1 ms
=
2.1 ms
```

So the consumer processes approximately:

```text
1 event every 2.1 ms
```

which is about:

```text
476 events / second
```

---

## Batched Consumer

Now suppose the consumer collects:

```text
4 events
```

before paying the fixed cost.

Then:

```text
batch processing time
=
2 ms
+
4 × 0.1 ms
=
2.4 ms
```

So the average cost per event becomes:

```text
2.4 ms / 4
=
0.6 ms per event
```

This is much better than:

```text
2.1 ms per event
```

without batching.

---

## Larger Batch Example

For a batch size of:

```text
16
```

the batch processing cost is:

```text
2 ms
+
16 × 0.1 ms
=
3.6 ms
```

Average cost per event:

```text
3.6 / 16
=
0.225 ms
```

For a batch size of:

```text
64
```

the batch processing cost is:

```text
2 ms
+
64 × 0.1 ms
=
8.4 ms
```

Average cost per event:

```text
8.4 / 64
≈
0.131 ms
```

Therefore:

```text
batch size increases
        ↓
fixed cost is amortized over more events
        ↓
effective cost per event decreases
        ↓
consumer throughput increases
```

---

## Experimental Pipeline

The new pipeline becomes:

```text
Producer
    ↓
BoundedQueue
    ↓
Consumer
    ↓
collect multiple events
    ↓
Batch
    ↓
simulate downstream processing
```

More specifically:

```text
Producer Thread
      ↓
BoundedQueue<ProfileEvent>
      ↓
Consumer Thread
      ↓
pop event
      ↓
append to current batch
      ↓
batch full?
   ┌──┴───┐
   │      │
  no     yes
   │      │
   │      ↓
   │   process batch
   │      ↓
   │   clear batch
   │
   └──────┘
```

---

## Fixed Workload

We will keep the workload constant:

```text
Total events:
1000

Queue capacity:
64

Producer delay:
0
```

The producer will generate events as quickly as possible.

We will compare:

```text
batch size = 1
batch size = 4
batch size = 16
batch size = 64
```

---

## Simulated Downstream Cost

To make batching meaningful, the consumer will simulate two types of cost:

```text
fixed batch overhead

per-event processing cost
```

For example:

```text
fixed overhead:
2 ms per batch

per-event cost:
0.1 ms per event
```

Therefore:

```text
processing time
=
fixed overhead
+
batch_size × per_event_cost
```

---

## Why We Need Both Costs

If the processing cost were only:

```text
0.1 ms per event
```

then batching would not significantly improve total work.

For example:

```text
1 event:
0.1 ms

16 events:
1.6 ms
```

The cost per event remains the same.

Batching becomes useful because a real downstream system often has a fixed cost.

For example:

```text
send()
```

has syscall overhead regardless of whether we send:

```text
1 event
```

or:

```text
16 events
```

Similarly, a database transaction has setup and commit overhead.

Batching reduces how often those fixed costs are paid.

---

## Batch Processing Model

For a batch containing `N` events:

```text
Batch Cost
=
Fixed Overhead
+
N × Per-Event Cost
```

With:

```text
Fixed Overhead = 2 ms
Per-Event Cost = 0.1 ms
```

we get:

| Batch Size | Batch Cost | Cost per Event |
|---:|---:|---:|
| 1 | 2.1 ms | 2.1 ms |
| 4 | 2.4 ms | 0.6 ms |
| 16 | 3.6 ms | 0.225 ms |
| 64 | 8.4 ms | 0.131 ms |

This predicts a strong throughput improvement as batch size increases.

---

## Expected Effect on Backpressure

Without efficient batching:

```text
consumer slow
    ↓
queue fills
    ↓
producer blocks
```

With larger batches:

```text
consumer processes events more efficiently
        ↓
queue drains faster
        ↓
queue fills less often
        ↓
producer blocking decreases
```

Therefore, we expect:

```text
batch size ↑
    ↓
consumer throughput ↑
    ↓
producer block count ↓
    ↓
producer blocked time ↓
```

---

## Expected Effect on Runtime

Because the consumer performs less fixed work per event, total runtime should decrease.

Conceptually:

```text
batch size 1
→ many downstream operations
→ high total overhead
→ long runtime
```

while:

```text
batch size 64
→ few downstream operations
→ lower amortized overhead
→ shorter runtime
```

---

## Number of Batches

The number of batches should approximately be:

```text
number_of_batches
=
ceil(total_events / batch_size)
```

For:

```text
total_events = 1000
```

we expect approximately:

| Batch Size | Number of Batches |
|---:|---:|
| 1 | 1000 |
| 4 | 250 |
| 16 | 63 |
| 64 | 16 |

The final batch may contain fewer events if the total number of events is not divisible by the batch size.

---

## Partial Final Batch

Suppose:

```text
total events = 10
batch size = 4
```

Then the batches are:

```text
Batch 1:
4 events

Batch 2:
4 events

Batch 3:
2 events
```

The consumer must still process the final partial batch.

Therefore, when the queue is closed and no more events remain:

```text
if current batch is not empty
    ↓
process final partial batch
```

Otherwise, those events would be lost.

---

## Important Tradeoff: Throughput vs Latency

Larger batches improve throughput, but they can increase waiting latency.

Suppose:

```text
batch size = 64
```

The first event in the batch may need to wait while the consumer collects the remaining events.

Conceptually:

```text
Event 1 arrives
    ↓
wait for event 2
    ↓
wait for event 3
    ↓
...
    ↓
wait for event 64
    ↓
process batch
```

So even though the batch is efficient:

```text
throughput improves
```

the first event may experience:

```text
higher latency
```

---

## Small Batch Tradeoff

A small batch:

```text
batch size = 1
```

provides:

```text
low batching delay
```

because events are processed immediately.

But:

```text
fixed overhead is paid for every event
```

so throughput is lower.

---

## Large Batch Tradeoff

A large batch:

```text
batch size = 64
```

provides:

```text
high throughput
```

because fixed overhead is amortized.

But it may cause:

```text
higher queueing latency
```

especially when the event arrival rate is low.

Therefore:

```text
larger batch
!=
always better
```

---

## Why This Matters for Profiling

Profiling systems often need both:

```text
high throughput
```

and:

```text
low reporting latency
```

For offline profiling:

```text
throughput
```

may be more important.

For example:

```text
large trace collection
post-processing
database ingestion
```

Large batches may be acceptable.

For live monitoring:

```text
latency
```

may matter more.

For example:

```text
live GPU utilization
interactive dashboards
real-time alerts
```

A smaller batch or timeout-based flush may be preferable.

---

## Metrics to Collect

For each batch size, we should measure:

```text
batch_size

number_of_batches

average_batch_size

produced_events

consumed_events

producer_block_count

producer_blocked_time

blocked_push_ratio

total_runtime

consumer_throughput
```

Later, we may also measure:

```text
event latency
```

to quantify the throughput-latency tradeoff.

---

## Average Batch Size

The average batch size can be calculated as:

```text
average_batch_size
=
consumed_events
---------------
number_of_batches
```

For example:

```text
1000 consumed events
63 batches

average batch size
≈ 15.87
```

This is slightly below 16 because the final batch may be partial.

---

## Consumer Throughput

Consumer throughput is:

```text
consumed_events
---------------
total_runtime
```

We expect this value to increase as batch size increases.

---

## Producer Blocking

The producer blocking metrics remain:

```text
producer_block_count

producer_blocked_time

blocked_push_ratio
```

If batching improves the consumer enough, these values should decrease significantly.

---

## Expected Experimental Trend

We expect something like:

```text
Batch Size = 1

many batches
↓
large fixed overhead
↓
slow consumer
↓
queue often full
↓
high producer blocking
```

```text
Batch Size = 4

fewer batches
↓
better amortization
↓
higher consumer throughput
↓
less producer blocking
```

```text
Batch Size = 16

much fewer batches
↓
higher efficiency
↓
significantly less backpressure
```

```text
Batch Size = 64

very few batches
↓
high throughput
↓
low producer blocking
↓
potentially higher batching latency
```

---

## Expected Results Table

The benchmark should eventually produce a table similar to:

| Batch Size | Batches | Avg Batch | Block Count | Block Time | Runtime | Throughput |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | ... | ... | ... | ... | ... | ... |
| 4 | ... | ... | ... | ... | ... | ... |
| 16 | ... | ... | ... | ... | ... | ... |
| 64 | ... | ... | ... | ... | ... | ... |

The main trends should be:

```text
batch size ↑
→ batch count ↓
→ cost per event ↓
→ runtime ↓
→ consumer throughput ↑
→ producer blocking ↓
```

---

## Connection to the Previous Serialization/Networking Work

Earlier we implemented:

```text
ProfileEvent
    ↓
Serializer
    ↓
FrameCodec
    ↓
BatchBuilder
    ↓
TCP
```

The current experiment uses a simplified simulated downstream cost so that we can isolate the effect of batching.

Later, the same idea can be applied to the real transport path:

```text
pop N events
    ↓
serialize N events
    ↓
build batch
    ↓
one TCP send
```

instead of:

```text
pop one event
    ↓
serialize
    ↓
send

repeat
```

So this experiment directly prepares us for realistic network batching.

---

## Key Profiling Insight

Backpressure does not always need to be handled only by:

```text
larger queues
```

or:

```text
dropping events
```

Sometimes the better solution is:

```text
make the downstream path more efficient
```

Batching is one way to do that.

Conceptually:

```text
Improve consumer efficiency
        ↓
increase service rate
        ↓
reduce queue buildup
        ↓
reduce backpressure
        ↓
reduce producer interference
```

This is fundamentally different from simply increasing queue capacity.

A larger queue:

```text
delays backpressure
```

whereas a more efficient consumer:

```text
can reduce the underlying bottleneck itself
```

---

## Questions to Answer After This Experiment

After completing this section, the following questions should be answerable:

- Why can batching improve downstream throughput?
- What is fixed per-batch overhead?
- What is per-event cost?
- Why does batching reduce average cost per event?
- How does batch size affect the number of downstream operations?
- Why can batching reduce producer backpressure?
- Why is batching fundamentally different from increasing queue capacity?
- Why must the final partial batch still be processed?
- What is the throughput-latency tradeoff?
- Why can a very large batch be bad for live profiling?
- Why might offline trace collection prefer larger batches?
- How would this experiment map to the real TCP profiling pipeline?

In [20]:
%%writefile distributed-profiler/benchmarks/batching_backpressure_benchmark.cpp

#include "bounded_queue.hpp"
#include "pipeline_metrics.hpp"
#include "profile_event.hpp"

#include <chrono>
#include <cstddef>
#include <cstdint>
#include <iomanip>
#include <iostream>
#include <stop_token>
#include <thread>
#include <vector>


// ============================================================
// Result of one batching experiment
// ============================================================

struct ExperimentResult {

    std::size_t batch_size = 0;

    std::size_t batch_count = 0;

    double average_batch_size = 0.0;

    PipelineMetrics metrics;

    double runtime_seconds = 0.0;

    double consumer_throughput = 0.0;

    double blocked_push_ratio = 0.0;

    double average_block_ms = 0.0;
};


// ============================================================
// Simulate downstream processing of one batch.
//
// Processing model:
//
// batch_cost
// =
// fixed_batch_overhead
// +
// batch_size * per_event_cost
// ============================================================

void process_batch(
    std::size_t batch_size,
    std::chrono::microseconds fixed_batch_overhead,
    std::chrono::microseconds per_event_cost
) {

    auto total_delay =
        fixed_batch_overhead
        +
        per_event_cost * batch_size;


    std::this_thread::sleep_for(
        total_delay
    );
}


// ============================================================
// Run one experiment for one batch size.
// ============================================================

ExperimentResult run_experiment(
    std::size_t batch_size,
    std::size_t total_events,
    std::size_t queue_capacity,
    std::chrono::microseconds fixed_batch_overhead,
    std::chrono::microseconds per_event_cost
) {

    using Clock =
        std::chrono::steady_clock;


    // ============================================================
    // Queue
    //
    // We use Block policy here because the goal is to measure
    // whether batching reduces backpressure.
    // ============================================================

    BoundedQueue<ProfileEvent> queue(
        queue_capacity,
        OverloadPolicy::Block
    );


    PipelineMetrics metrics;


    std::size_t batch_count = 0;


    auto experiment_start =
        Clock::now();


    // ============================================================
    // Producer
    // ============================================================

    std::jthread producer(

        [&](std::stop_token stop_token) {

            for (
                std::size_t i = 0;
                i < total_events;
                ++i
            ) {

                ++metrics.attempted_events;


                ProfileEvent event =
                    KernelEvent{
                        static_cast<std::uint64_t>(i),
                        100,
                        0,
                        "synthetic_kernel"
                    };


                PushResult result =
                    queue.push(
                        std::move(event),
                        stop_token
                    );


                if (!result.success) {
                    break;
                }


                ++metrics.produced_events;


                if (result.blocked) {

                    ++metrics.producer_block_count;

                    metrics.producer_blocked_time +=
                        result.wait_time;
                }


                std::size_t current_size =
                    queue.size();


                if (
                    current_size >
                    metrics.max_queue_occupancy
                ) {

                    metrics.max_queue_occupancy =
                        current_size;
                }
            }


            queue.close();
        }
    );



    // ============================================================
    // Consumer
    //
    // Pop events and accumulate them into a local batch.
    //
    // Downstream cost is paid once per batch instead of once
    // per event.
    // ============================================================

    std::jthread consumer(

        [&](std::stop_token stop_token) {

            std::vector<ProfileEvent> batch;


            batch.reserve(
                batch_size
            );


            while (true) {

                auto event =
                    queue.pop(
                        stop_token
                    );


                // ------------------------------------------------
                // Queue is closed and empty.
                // ------------------------------------------------

                if (!event.has_value()) {
                    break;
                }


                // ------------------------------------------------
                // Add event to current batch.
                // ------------------------------------------------

                batch.push_back(
                    std::move(*event)
                );


                // ------------------------------------------------
                // Full batch available.
                // ------------------------------------------------

                if (
                    batch.size() ==
                    batch_size
                ) {

                    process_batch(
                        batch.size(),
                        fixed_batch_overhead,
                        per_event_cost
                    );


                    metrics.consumed_events +=
                        batch.size();


                    ++batch_count;


                    batch.clear();
                }
            }


            // ====================================================
            // Process final partial batch.
            //
            // Example:
            //
            // 1000 events
            // batch size = 64
            //
            // 15 full batches = 960
            // final batch = 40
            // ====================================================

            if (!batch.empty()) {

                process_batch(
                    batch.size(),
                    fixed_batch_overhead,
                    per_event_cost
                );


                metrics.consumed_events +=
                    batch.size();


                ++batch_count;


                batch.clear();
            }
        }
    );


    // ============================================================
    // Wait for both threads
    // ============================================================

    producer.join();

    consumer.join();


    auto experiment_end =
        Clock::now();


    // ============================================================
    // Runtime.
    // ============================================================

    metrics.total_runtime =
        std::chrono::duration_cast<
            std::chrono::nanoseconds
        >(
            experiment_end -
            experiment_start
        );



    // ============================================================
    // Derived metrics.
    // ============================================================

    double runtime_seconds =
        std::chrono::duration<double>(
            metrics.total_runtime
        ).count();


    double consumer_throughput = 0.0;

    if (runtime_seconds > 0.0) {

        consumer_throughput =
            static_cast<double>(
                metrics.consumed_events
            )
            /
            runtime_seconds;
    }


    double blocked_push_ratio = 0.0;

    if (
        metrics.attempted_events > 0
    ) {

        blocked_push_ratio =
            100.0 *
            static_cast<double>(
                metrics.producer_block_count
            )
            /
            static_cast<double>(
                metrics.attempted_events
            );
    }


    double average_block_ms = 0.0;

    if (
        metrics.producer_block_count > 0
    ) {

        average_block_ms =
            std::chrono::duration<
                double,
                std::milli
            >(
                metrics.producer_blocked_time
            ).count()
            /
            static_cast<double>(
                metrics.producer_block_count
            );
    }


    double average_batch_size = 0.0;

    if (
        batch_count > 0
    ) {

        average_batch_size =
            static_cast<double>(
                metrics.consumed_events
            )
            /
            static_cast<double>(
                batch_count
            );
    }



    // ============================================================
    // Build result.
    // ============================================================

    ExperimentResult result;


    result.batch_size =
        batch_size;


    result.batch_count =
        batch_count;


    result.average_batch_size =
        average_batch_size;


    result.metrics =
        metrics;


    result.runtime_seconds =
        runtime_seconds;


    result.consumer_throughput =
        consumer_throughput;


    result.blocked_push_ratio =
        blocked_push_ratio;


    result.average_block_ms =
        average_block_ms;


    return result;
}



// ============================================================
// main()
// ============================================================

int main() {

    // ============================================================
    // Fixed workload
    // ============================================================

    const std::size_t total_events =
        1000;


    const std::size_t queue_capacity =
        64;


    // ============================================================
    // Simulated downstream cost
    //
    // 2 ms fixed batch cost
    //
    // 0.1 ms = 100 us per event
    // ============================================================

    const auto fixed_batch_overhead =
        std::chrono::milliseconds(2);


    const auto per_event_cost =
        std::chrono::microseconds(100);


    // ============================================================
    // Batch sizes to compare
    // ============================================================

    const std::vector<std::size_t>
        batch_sizes = {

            1,
            4,
            16,
            64
        };


    std::vector<ExperimentResult>
        results;


    results.reserve(
        batch_sizes.size()
    );


    // ============================================================
    // Run experiments.
    // ============================================================

    for (
        std::size_t batch_size :
        batch_sizes
    ) {

        ExperimentResult result =
            run_experiment(
                batch_size,
                total_events,
                queue_capacity,
                fixed_batch_overhead,
                per_event_cost
            );


        results.push_back(
            std::move(result)
        );
    }


    // ============================================================
    // Print summary
    // ============================================================

    std::cout
        << "\n=== Batching + Backpressure ===\n\n";


    std::cout
        << std::left

        << std::setw(12)
        << "Batch"

        << std::setw(12)
        << "Batches"

        << std::setw(14)
        << "Avg Batch"

        << std::setw(12)
        << "Blocks"

        << std::setw(14)
        << "Block Ratio"

        << std::setw(14)
        << "Avg Block"

        << std::setw(14)
        << "Runtime"

        << std::setw(18)
        << "Throughput"

        << '\n';


    std::cout
        << std::string(
            110,
            '-'
        )
        << '\n';


    for (
        const auto& result :
        results
    ) {

        std::cout
            << std::left

            << std::setw(12)
            << result.batch_size

            << std::setw(12)
            << result.batch_count

            << std::setw(14)
            << result.average_batch_size

            << std::setw(12)
            << result.metrics.producer_block_count

            << std::setw(14)
            << result.blocked_push_ratio

            << std::setw(14)
            << result.average_block_ms

            << std::setw(14)
            << result.runtime_seconds

            << std::setw(18)
            << result.consumer_throughput

            << '\n';
    }


    return 0;
}

Writing distributed-profiler/benchmarks/batching_backpressure_benchmark.cpp


In [21]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/benchmarks/batching_backpressure_benchmark.cpp \
    -pthread \
    -o batching_backpressure_benchmark

In [22]:
!./batching_backpressure_benchmark


=== Batching + Backpressure ===

Batch       Batches     Avg Batch     Blocks      Block Ratio   Avg Block     Runtime       Throughput        
--------------------------------------------------------------------------------------------------------------
1           1000        1             935         93.5          2.22551       2.23116       448.198           
4           250         4             233         23.3          2.53829       0.638302      1566.66           
16          63          15.873        60          6             3.65984       0.241292      4144.36           
64          16          62.5          15          1.5           8.03255       0.140538      7115.51           


## Slow Network / Collector Simulation

### Goal

The previous experiments studied backpressure inside the local producer-consumer pipeline.

We showed that:

```text
small queue
→ producer blocks earlier
```

```text
larger queue
→ absorbs more burst traffic
→ but does not remove the bottleneck
```

```text
drop policies
→ reduce producer blocking
→ but lose profiling data
```

and:

```text
batching
→ improves downstream efficiency
→ increases consumer throughput
→ reduces backpressure
```

Now we want to connect these ideas to the distributed profiling architecture.

The central question is:

> What happens when the network or remote collector becomes slower than the profiling producer?

---

## Distributed Profiling Path

The complete profiling path is:

```text
Application / GPU
        ↓
Profiling Event Producer
        ↓
BoundedQueue
        ↓
Batching Consumer
        ↓
TCP Agent
        ↓
Network
        ↓
Collector
        ↓
Storage / Analysis
```

Backpressure can originate anywhere downstream.

For example:

```text
slow database
    ↓
slow collector
    ↓
collector reads network more slowly
    ↓
sender cannot transmit as quickly
    ↓
local consumer slows down
    ↓
queue fills
    ↓
producer blocks
```

This is called:

```text
backpressure propagation
```

---

## Why Simulate the Network First?

A complete multi-machine benchmark would require:

```text
multiple hosts
real network variability
collector deployment
remote storage
```

That is not necessary to understand the core behavior.

Instead, we can simulate downstream delay locally.

Conceptually:

```text
consumer builds batch
        ↓
simulate network / collector delay
        ↓
consumer cannot pop more events temporarily
        ↓
queue starts filling
```

This lets us isolate the relationship between:

```text
downstream service time
```

and:

```text
upstream backpressure
```

---

## Experimental Pipeline

The experiment will use:

```text
Fast Producer
      ↓
BoundedQueue
      ↓
Batching Consumer
      ↓
Simulated Network / Collector
```

The consumer will:

```text
pop several events
      ↓
form a batch
      ↓
simulate send / collector delay
      ↓
continue
```

---

## Fixed Workload

We will keep several parameters constant.

For example:

```text
Total events:
1000

Queue capacity:
64

Batch size:
16

Producer delay:
0
```

The producer generates events as quickly as possible.

The only variable will be:

```text
downstream delay
```

---

## Downstream Delay Cases

We can compare:

```text
0 ms

1 ms

5 ms

10 ms

20 ms
```

per batch.

Conceptually:

```text
0 ms
→ very fast downstream

1 ms
→ small network / collector cost

5 ms
→ moderate downstream slowdown

10 ms
→ slow downstream

20 ms
→ severe downstream bottleneck
```

---

## Simulated Consumer Behavior

For every batch:

```text
collect batch
    ↓
process / serialize
    ↓
simulate downstream delay
    ↓
continue
```

For example:

```cpp
std::this_thread::sleep_for(
    downstream_delay
);
```

This delay represents the combined effect of things such as:

```text
TCP transmission

network congestion

collector processing

database insertion

disk flushing
```

We are not claiming that all of those costs are literally `sleep_for()`.

The sleep is simply a controlled model of slower downstream service.

---

## Expected Behavior with Fast Downstream

Suppose:

```text
downstream delay = 0 ms
```

Then the consumer can drain batches quickly.

Conceptually:

```text
producer
    ↓
queue
    ↓
consumer drains quickly
```

We expect:

```text
low queue pressure

few producer blocks

low blocked time

high throughput
```

---

## Expected Behavior with Moderate Delay

Suppose:

```text
downstream delay = 5 ms per batch
```

The consumer now spends more time away from the queue.

During that period:

```text
producer continues pushing
```

so:

```text
queue occupancy rises
```

Eventually:

```text
queue may become full
```

and:

```text
producer starts blocking
```

---

## Expected Behavior with Severe Delay

Suppose:

```text
downstream delay = 20 ms per batch
```

Then:

```text
consumer removes batch
        ↓
consumer waits 20 ms
        ↓
producer rapidly refills queue
        ↓
queue becomes full
        ↓
producer blocks
```

This repeats for every batch.

The system enters a steady-state pattern similar to:

```text
consumer frees several slots
        ↓
producer fills them
        ↓
queue full
        ↓
producer waits
        ↓
consumer finishes downstream work
        ↓
consumer frees more slots
```

---

## Backpressure Propagation

The important point is that the producer does not need to know why the downstream system is slow.

It only observes:

```text
queue has no free capacity
```

The actual root cause may be:

```text
slow TCP send
slow network
slow collector
slow database
slow storage
```

but the effect propagates upstream:

```text
downstream slowdown
        ↓
consumer slowdown
        ↓
queue buildup
        ↓
producer blocking
```

This decoupling is one reason bounded queues are useful in asynchronous systems.

---

## Relationship to TCP Flow Control

In a real TCP pipeline, the behavior can be more indirect.

Conceptually:

```text
Collector reads slowly
        ↓
receiver socket buffer fills
        ↓
TCP flow control reduces sender progress
        ↓
sender-side send() may block or slow down
        ↓
profiling consumer thread slows
        ↓
local queue fills
        ↓
producer experiences backpressure
```

So backpressure can propagate across process and machine boundaries.

Our simulation models the resulting slower consumer service rate.

---

## Why Batching Still Matters

Suppose the network has a fixed per-send cost.

Without batching:

```text
event
↓
send

event
↓
send

event
↓
send
```

The network overhead is paid repeatedly.

With batching:

```text
event
event
event
event
    ↓
one batch
    ↓
one send
```

This can reduce downstream overhead.

Therefore:

```text
batching
```

can partially protect the pipeline from downstream cost.

However, if the collector is sufficiently slow:

```text
no batch size can completely eliminate the bottleneck
```

because the average downstream service rate is still lower than the producer rate.

---

## Important Queue Insight

This experiment reinforces the difference between:

```text
buffering
```

and:

```text
service capacity
```

A larger queue can absorb a temporary slowdown:

```text
short burst of network delay
        ↓
queue buffers events
        ↓
producer may continue temporarily
```

But if the downstream remains slow for a long time:

```text
queue eventually fills
```

regardless of capacity.

Therefore:

> A queue can absorb temporary imbalance, but it cannot solve a sustained throughput mismatch.

---

## Metrics to Collect

For each downstream delay, we should measure:

```text
downstream_delay

number_of_batches

produced_events

consumed_events

producer_block_count

producer_blocked_time

blocked_push_ratio

average_block_time

max_queue_occupancy

max_queue_utilization

total_runtime

consumer_throughput
```

These metrics allow us to see how downstream slowdown propagates into the local profiling pipeline.

---

## Expected Trends

We expect:

```text
downstream delay ↑
        ↓
consumer service rate ↓
        ↓
runtime ↑
        ↓
queue occupancy ↑
        ↓
producer block count ↑
        ↓
producer blocked time ↑
        ↓
throughput ↓
```

---

## Expected Results Table

The final benchmark may look like:

| Downstream Delay | Blocks | Block Ratio | Block Time | Max Queue | Runtime | Throughput |
|---:|---:|---:|---:|---:|---:|---:|
| 0 ms | ... | ... | ... | ... | ... | ... |
| 1 ms | ... | ... | ... | ... | ... | ... |
| 5 ms | ... | ... | ... | ... | ... | ... |
| 10 ms | ... | ... | ... | ... | ... | ... |
| 20 ms | ... | ... | ... | ... | ... | ... |

The exact values depend on scheduling, but the overall trend should be clear.

---

## Temporary Slowdown vs Sustained Slowdown

A useful distinction is:

```text
temporary overload
```

versus:

```text
sustained overload
```

### Temporary Slowdown

Example:

```text
collector pauses briefly
```

A sufficiently large queue may absorb the burst.

After the collector recovers:

```text
consumer drains backlog
```

and the system returns to normal.

---

### Sustained Slowdown

Example:

```text
producer produces 10,000 events/s

collector can process only 2,000 events/s
```

Then the difference is:

```text
8,000 events/s
```

The queue keeps accumulating backlog until:

```text
queue becomes full
```

At that point, the overload policy determines what happens.

---

## Connection to Overload Policies

Once the queue becomes full, we return to the previous design decision.

With:

```text
Block
```

the slowdown propagates directly to the producer.

With:

```text
DropNewest
```

new events are lost.

With:

```text
DropOldest
```

old buffered events are replaced.

Therefore the complete overload chain becomes:

```text
Slow Collector
      ↓
Slow Consumer
      ↓
Queue Full
      ↓
Overload Policy
      │
      ├── Block producer
      │
      ├── Drop newest
      │
      └── Drop oldest
```

The previous experiments and this experiment are therefore part of the same system behavior.

---

## Profiling-System Interpretation

For a profiling infrastructure, downstream slowdown may be caused by:

```text
network congestion

remote collector overload

slow storage

database contention

large trace volume

multiple profiling clients
```

A well-designed profiler should remain predictable under these conditions.

It should answer:

```text
Will the application block?

Will profiling data be dropped?

How much buffering is available?

How is overload reported?

Can the collector recover after a temporary slowdown?
```

These are production-system concerns, not only implementation details.

---

## Observability

In a production system, the profiler should expose internal metrics such as:

```text
current queue occupancy

maximum queue occupancy

drop count

blocked producer time

batch count

network send latency

collector processing latency
```

Without observability, a user may only see:

```text
profiling became slow
```

without knowing whether the cause was:

```text
producer
queue
network
collector
storage
```

Instrumentation helps locate the actual bottleneck.

---

## Failure Isolation

The queue also acts as a boundary between components.

For example:

```text
Producer
    ↓
Queue
    ↓
Network Consumer
```

If the network temporarily slows down, the producer does not immediately need to understand or handle the network failure.

Instead:

```text
queue absorbs temporary imbalance
```

until its capacity is exhausted.

This provides some isolation between:

```text
event generation
```

and:

```text
event transport
```

---

## Key System Insight

The most important lesson from this experiment is:

> Backpressure is an end-to-end system property.

A slow collector can eventually affect the profiling producer even though they are separated by:

```text
threads

queues

TCP

processes

possibly different machines
```

The slowdown propagates through every bounded stage of the pipeline.

---

## Connection to the Real TCP Implementation

In the earlier networking work, we implemented:

```text
BatchBuilder
    ↓
send_all()
    ↓
TCP
    ↓
recv_exact()
    ↓
BatchParser
```

The current experiment models the situation where that transport path becomes slower.

A future end-to-end benchmark can replace:

```cpp
sleep_for(downstream_delay)
```

with:

```text
real serialize
real batch
real send_all
real collector recv
```

while keeping the same queue and metrics infrastructure.

---

## Day 6 Final System Picture

After this experiment, the complete backpressure story becomes:

```text
Fast Profiling Producer
        ↓
BoundedQueue
        ↓
Batching Consumer
        ↓
TCP / Network
        ↓
Collector / Storage
```

If downstream is fast:

```text
queue remains manageable
↓
little producer interference
```

If downstream slows:

```text
queue fills
↓
backpressure propagates upstream
```

Then the system can respond through:

```text
queue sizing

batching

blocking

dropping

downstream optimization
```

---

## Questions to Answer After This Experiment

After completing this section, the following questions should be answerable:

- How can a slow collector create backpressure in the producer?
- How does backpressure propagate through an asynchronous pipeline?
- Why can a queue absorb temporary overload but not sustained overload?
- What is the difference between buffering capacity and service capacity?
- How can TCP flow control eventually affect the profiling producer?
- Why can batching reduce network-related backpressure?
- Why can batching not solve an arbitrarily slow collector?
- How do overload policies interact with downstream slowdown?
- What metrics would you expose to diagnose backpressure in production?
- Why is backpressure an end-to-end property rather than only a queue problem?
- How does this experiment connect the local C++ queue implementation with the TCP agent/collector architecture?

In [23]:
%%writefile distributed-profiler/benchmarks/slow_downstream_benchmark.cpp

#include "bounded_queue.hpp"
#include "pipeline_metrics.hpp"
#include "profile_event.hpp"

#include <chrono>
#include <cstddef>
#include <cstdint>
#include <iomanip>
#include <iostream>
#include <stop_token>
#include <thread>
#include <vector>


// ============================================================
// Result of one downstream-delay experiment
// ============================================================

struct ExperimentResult {

    std::chrono::milliseconds downstream_delay{0};

    std::size_t batch_count = 0;

    double average_batch_size = 0.0;

    PipelineMetrics metrics;

    double runtime_seconds = 0.0;

    double consumer_throughput = 0.0;

    double blocked_push_ratio = 0.0;

    double average_block_ms = 0.0;

    double total_blocked_seconds = 0.0;

    double max_queue_utilization = 0.0;
};


// ============================================================
// Simulate downstream network / collector work.
//
// In a real system, this could represent:
//
// serialize batch
//      ↓
// send_all()
//      ↓
// TCP/network delay
//      ↓
// collector processing
//      ↓
// database/storage
//
// Here we use sleep_for() so the slowdown is controlled.
// ============================================================

void process_downstream(
    std::chrono::milliseconds downstream_delay
) {

    std::this_thread::sleep_for(
        downstream_delay
    );
}


// ============================================================
// Run one experiment for one downstream delay.
// ============================================================

ExperimentResult run_experiment(
    std::chrono::milliseconds downstream_delay,
    std::size_t total_events,
    std::size_t queue_capacity,
    std::size_t batch_size
) {

    using Clock =
        std::chrono::steady_clock;


    // ============================================================
    // Use Block policy.
    //
    // This lets us observe how downstream slowdown propagates
    // back to the producer.
    // ============================================================

    BoundedQueue<ProfileEvent> queue(
        queue_capacity,
        OverloadPolicy::Block
    );


    PipelineMetrics metrics;


    std::size_t batch_count = 0;


    auto experiment_start =
        Clock::now();


    // ============================================================
    // Producer
    //
    // Generate events as fast as possible.
    // ============================================================

    std::jthread producer(

        [&](std::stop_token stop_token) {

            for (
                std::size_t i = 0;
                i < total_events;
                ++i
            ) {

                ++metrics.attempted_events;


                ProfileEvent event =
                    KernelEvent{
                        static_cast<std::uint64_t>(i),
                        100,
                        0,
                        "synthetic_kernel"
                    };


                PushResult result =
                    queue.push(
                        std::move(event),
                        stop_token
                    );


                if (!result.success) {
                    break;
                }


                ++metrics.produced_events;


                // ------------------------------------------------
                // Backpressure metrics
                // ------------------------------------------------

                if (result.blocked) {

                    ++metrics.producer_block_count;

                    metrics.producer_blocked_time +=
                        result.wait_time;
                }


                // ------------------------------------------------
                // Queue occupancy
                // ------------------------------------------------

                std::size_t current_size =
                    queue.size();


                if (
                    current_size >
                    metrics.max_queue_occupancy
                ) {

                    metrics.max_queue_occupancy =
                        current_size;
                }
            }


            queue.close();
        }
    );


    // ============================================================
    // Consumer
    //
    // Consumer drains events in batches.
    //
    // After collecting one batch, it simulates a downstream
    // network / collector delay.
    // ============================================================

    std::jthread consumer(

        [&](std::stop_token stop_token) {

            std::vector<ProfileEvent> batch;


            batch.reserve(
                batch_size
            );


            while (true) {

                auto event =
                    queue.pop(
                        stop_token
                    );


                // ------------------------------------------------
                // Queue closed and empty.
                // ------------------------------------------------

                if (!event.has_value()) {
                    break;
                }


                batch.push_back(
                    std::move(*event)
                );


                // ------------------------------------------------
                // Full batch ready.
                // ------------------------------------------------

                if (
                    batch.size() ==
                    batch_size
                ) {

                    // Simulate slow downstream path.
                    process_downstream(
                        downstream_delay
                    );


                    metrics.consumed_events +=
                        batch.size();


                    ++batch_count;


                    batch.clear();
                }
            }


            // ====================================================
            // Process final partial batch.
            // ====================================================

            if (!batch.empty()) {

                process_downstream(
                    downstream_delay
                );


                metrics.consumed_events +=
                    batch.size();


                ++batch_count;


                batch.clear();
            }
        }
    );


    // ============================================================
    // Wait for completion.
    // ============================================================

    producer.join();

    consumer.join();


    auto experiment_end =
        Clock::now();


    // ============================================================
    // Runtime
    // ============================================================

    metrics.total_runtime =
        std::chrono::duration_cast<
            std::chrono::nanoseconds
        >(
            experiment_end -
            experiment_start
        );


    // ============================================================
    // Derived metrics
    // ============================================================

    double runtime_seconds =
        std::chrono::duration<double>(
            metrics.total_runtime
        ).count();


    double consumer_throughput = 0.0;

    if (runtime_seconds > 0.0) {

        consumer_throughput =
            static_cast<double>(
                metrics.consumed_events
            )
            /
            runtime_seconds;
    }


    double blocked_push_ratio = 0.0;

    if (
        metrics.attempted_events > 0
    ) {

        blocked_push_ratio =
            100.0 *
            static_cast<double>(
                metrics.producer_block_count
            )
            /
            static_cast<double>(
                metrics.attempted_events
            );
    }


    double average_block_ms = 0.0;

    if (
        metrics.producer_block_count > 0
    ) {

        average_block_ms =
            std::chrono::duration<
                double,
                std::milli
            >(
                metrics.producer_blocked_time
            ).count()
            /
            static_cast<double>(
                metrics.producer_block_count
            );
    }


    double total_blocked_seconds =
        std::chrono::duration<double>(
            metrics.producer_blocked_time
        ).count();


    double average_batch_size = 0.0;

    if (
        batch_count > 0
    ) {

        average_batch_size =
            static_cast<double>(
                metrics.consumed_events
            )
            /
            static_cast<double>(
                batch_count
            );
    }


    double max_queue_utilization = 0.0;

    if (
        queue_capacity > 0
    ) {

        max_queue_utilization =
            100.0 *
            static_cast<double>(
                metrics.max_queue_occupancy
            )
            /
            static_cast<double>(
                queue_capacity
            );
    }


    // ============================================================
    // Build result
    // ============================================================

    ExperimentResult result;


    result.downstream_delay =
        downstream_delay;


    result.batch_count =
        batch_count;


    result.average_batch_size =
        average_batch_size;


    result.metrics =
        metrics;


    result.runtime_seconds =
        runtime_seconds;


    result.consumer_throughput =
        consumer_throughput;


    result.blocked_push_ratio =
        blocked_push_ratio;


    result.average_block_ms =
        average_block_ms;


    result.total_blocked_seconds =
        total_blocked_seconds;


    result.max_queue_utilization =
        max_queue_utilization;


    return result;
}


// ============================================================
// main
// ============================================================

int main() {

    // ============================================================
    // Fixed workload
    // ============================================================

    const std::size_t total_events =
        1000;


    const std::size_t queue_capacity =
        64;


    const std::size_t batch_size =
        16;


    // ============================================================
    // Downstream delays to compare
    // ============================================================

    const std::vector<
        std::chrono::milliseconds
    > downstream_delays = {

        std::chrono::milliseconds(0),

        std::chrono::milliseconds(1),

        std::chrono::milliseconds(5),

        std::chrono::milliseconds(10),

        std::chrono::milliseconds(20)
    };


    std::vector<ExperimentResult>
        results;


    results.reserve(
        downstream_delays.size()
    );


    // ============================================================
    // Run experiments
    // ============================================================

    for (
        auto delay :
        downstream_delays
    ) {

        ExperimentResult result =
            run_experiment(
                delay,
                total_events,
                queue_capacity,
                batch_size
            );


        results.push_back(
            std::move(result)
        );
    }


    // ============================================================
    // Print summary
    // ============================================================

    std::cout
        << "\n=== Slow Network / Collector Simulation ===\n\n";


    std::cout
        << std::left

        << std::setw(12)
        << "Delay(ms)"

        << std::setw(12)
        << "Batches"

        << std::setw(14)
        << "Avg Batch"

        << std::setw(12)
        << "Blocks"

        << std::setw(14)
        << "Block Ratio"

        << std::setw(14)
        << "Avg Block"

        << std::setw(16)
        << "Total Block"

        << std::setw(14)
        << "Max Queue%"

        << std::setw(14)
        << "Runtime"

        << std::setw(18)
        << "Throughput"

        << '\n';


    std::cout
        << std::string(
            138,
            '-'
        )
        << '\n';


    for (
        const auto& result :
        results
    ) {

        std::cout
            << std::left

            << std::setw(12)
            << result.downstream_delay.count()

            << std::setw(12)
            << result.batch_count

            << std::setw(14)
            << result.average_batch_size

            << std::setw(12)
            << result.metrics.producer_block_count

            << std::setw(14)
            << result.blocked_push_ratio

            << std::setw(14)
            << result.average_block_ms

            << std::setw(16)
            << result.total_blocked_seconds

            << std::setw(14)
            << result.max_queue_utilization

            << std::setw(14)
            << result.runtime_seconds

            << std::setw(18)
            << result.consumer_throughput

            << '\n';
    }


    return 0;
}

Writing distributed-profiler/benchmarks/slow_downstream_benchmark.cpp


In [24]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/benchmarks/slow_downstream_benchmark.cpp \
    -pthread \
    -o slow_downstream_benchmark

In [25]:
!./slow_downstream_benchmark


=== Slow Network / Collector Simulation ===

Delay(ms)   Batches     Avg Batch     Blocks      Block Ratio   Avg Block     Total Block     Max Queue%    Runtime       Throughput        
------------------------------------------------------------------------------------------------------------------------------------------
0           63          15.873        4           0.4           0.25984       0.00103936      100           0.00868537    115136            
1           63          15.873        69          6.9           0.943306      0.0650881       100           0.0748717     13356.2           
5           63          15.873        60          6             4.96473       0.297884        100           0.327505      3053.39           
10          63          15.873        62          6.2           9.50473       0.589293        100           0.643282      1554.53           
20          63          15.873        60          6             19.5243       1.17146         100           1.

# Backpressure and Scalability — Experiment Summary

## Objective

This notebook studied how a profiling pipeline behaves when event production is faster than downstream processing.

The main goals were to understand:

- how bounded queues create backpressure,
- how queue capacity affects producer blocking,
- how different overload policies trade data completeness for lower producer interference,
- how batching improves downstream efficiency,
- and how slowdown in the network or collector propagates back to the profiling producer.

The experiments used a simplified profiling pipeline:

```text
Fast Producer
      ↓
BoundedQueue<ProfileEvent>
      ↓
Consumer
      ↓
Batching / Simulated Downstream Work
```

The queue used C++20 synchronization primitives and supported precise measurements of:

```text
producer block count
producer blocked time
average block duration
queue occupancy
drop rate
runtime
throughput
```

---

# Backpressure Baseline

The first experiment used:

```text
Total events:      100
Queue capacity:    8
Consumer delay:    10 ms/event
```

The producer generated events as fast as possible while the consumer intentionally processed one event every 10 ms.

Measured result:

```text
Attempted events:          100
Produced events:           100
Consumed events:           100
Dropped events:              0

Queue capacity:              8
Max queue occupancy:         8
Max queue utilization:     100%

Producer block count:       92
Producer blocked time:   0.924844 s
Producer blocked percent: 90.939%
Average block time:      10.0527 ms
Blocked push ratio:        92%

Total runtime:           1.01699 s
Producer throughput:     98.329 events/s
Consumer throughput:     98.329 events/s
```

The most important observation was:

```text
Average producer block time ≈ 10 ms
Consumer processing delay  = 10 ms
```

This shows that once the queue became full, the producer typically had to wait until the consumer processed one event and freed one queue slot.

The producer and consumer had similar measured throughput not because the producer was naturally slow, but because backpressure throttled the producer down to the consumer's service rate.

---

# Precise Blocking Instrumentation

Initially, blocking time was approximated by timing the entire `push()` operation.

That included:

```text
mutex acquisition
condition-variable waiting
queue insertion
function overhead
```

The queue was then instrumented internally so that it could report:

```cpp
struct PushResult {
    bool success;
    bool blocked;
    bool dropped;

    std::chrono::nanoseconds wait_time;
};
```

The new implementation measured only the time spent waiting because the queue was full.

This provided much more meaningful backpressure metrics:

```text
producer_block_count
producer_blocked_time
average_block_time
blocked_push_ratio
```

The measured average block time closely matching the consumer delay validated the instrumentation.

---

# Queue Capacity Experiment

The next experiment studied how queue capacity affects backpressure.

The workload remained fixed:

```text
Total events:      100
Consumer delay:    10 ms/event
Producer delay:    0
```

Only queue capacity changed:

```text
1
8
16
32
64
128
```

Measured results:

| Capacity | Blocks | Block Ratio | Block Time (s) | Avg Block (ms) | Max Queue | Queue Utilization | Runtime (s) |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 99 | 99% | 0.999509 | 10.0961 | 1 | 100% | 1.02075 |
| 8 | 91 | 91% | 0.923681 | 10.1503 | 8 | 100% | 1.01592 |
| 16 | 83 | 83% | 0.846137 | 10.1944 | 16 | 100% | 1.01956 |
| 32 | 67 | 67% | 0.680501 | 10.1567 | 32 | 100% | 1.01610 |
| 64 | 35 | 35% | 0.355952 | 10.1701 | 64 | 100% | 1.01643 |
| 128 | 0 | 0% | 0 | 0 | 99 | 77.343% | 1.01627 |

Throughput remained almost unchanged:

```text
Capacity 1   → 97.97 events/s
Capacity 8   → 98.43 events/s
Capacity 16  → 98.08 events/s
Capacity 32  → 98.42 events/s
Capacity 64  → 98.38 events/s
Capacity 128 → 98.40 events/s
```

The experiment demonstrated:

```text
queue capacity ↑
        ↓
more burst buffering
        ↓
producer blocking ↓
```

However:

```text
consumer throughput ≈ unchanged
total runtime ≈ unchanged
```

because the consumer remained the bottleneck.

The main conclusion was:

> Increasing queue capacity delays backpressure, but it does not increase downstream processing capacity.

A large queue can absorb a temporary burst, but it cannot solve sustained producer-consumer imbalance.

---

# Overload Policy Comparison

Three overload policies were implemented:

```text
Block Producer
Drop Newest
Drop Oldest
```

The fixed workload was:

```text
Total events:      100
Queue capacity:    8
Consumer delay:    10 ms/event
```

---

## Block Producer

Consumed event IDs:

```text
0 1 2 3 ... 99
```

Measured behavior:

```text
Attempted:     100
Produced:      100
Consumed:      100
Dropped:         0

Blocks:         91
Avg Block:   10.1377 ms
Runtime:      1.01498 s
```

This policy preserved the complete event stream.

Its tradeoff was high producer interference:

```text
slow consumer
    ↓
queue full
    ↓
producer waits
```

This policy is appropriate when complete trace fidelity is required.

---

## Drop Newest

Consumed event IDs:

```text
0 1 2 3 4 5 6 7
```

Measured behavior:

```text
Attempted:      100
Produced:         8
Consumed:         8
Dropped:         92
Drop Rate:       92%

Blocks:           0
Runtime:      ~0.082 s
```

Once the queue filled, incoming events were discarded.

The queue preserved older buffered data:

```text
[0][1][2][3][4][5][6][7]
```

while newer events were lost.

This eliminated producer blocking but sacrificed recent profiling information.

---

## Drop Oldest

Consumed event IDs:

```text
92 93 94 95 96 97 98 99
```

When a new event arrived to a full queue:

```text
oldest buffered event
        ↓
dropped

new event
        ↓
inserted
```

This preserved the newest profiling information while discarding stale queued events.

The experiment clearly demonstrated the semantic difference:

```text
DropNewest
→ preserves old backlog

DropOldest
→ preserves newest data
```

This makes Drop Oldest attractive for use cases such as:

```text
live telemetry
current utilization
health monitoring
real-time dashboards
```

while both drop policies are unsuitable for exact lossless tracing.

---

# Overload Policy Tradeoff

The three policies optimize different objectives:

| Policy | Producer Blocking | Data Loss | Preserves Old Data | Preserves New Data |
|---|---|---|---|---|
| Block | High | No | Yes | Yes |
| Drop Newest | None | Yes | Yes | No |
| Drop Oldest | None | Yes | No | Yes |

The central profiling tradeoff is:

```text
data completeness
        vs
application interference
```

A profiler should not automatically assume that preserving every event is always the best design, because blocking the application may significantly perturb the workload being measured.

---

# Batching + Backpressure

The next experiment tested whether improving consumer efficiency could reduce the underlying bottleneck.

The workload was:

```text
Total events:      1000
Queue capacity:    64
Producer delay:    0
```

The downstream processing model was:

```text
Batch cost
=
2 ms fixed overhead
+
0.1 ms × number of events
```

Batch sizes tested:

```text
1
4
16
64
```

Measured results:

| Batch Size | Batches | Avg Batch | Blocks | Block Ratio | Avg Block (ms) | Runtime (s) | Throughput (events/s) |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 1000 | 1 | 935 | 93.5% | 2.22551 | 2.23116 | 448.198 |
| 4 | 250 | 4 | 233 | 23.3% | 2.53829 | 0.638302 | 1566.66 |
| 16 | 63 | 15.873 | 60 | 6.0% | 3.65984 | 0.241292 | 4144.36 |
| 64 | 16 | 62.5 | 15 | 1.5% | 8.03255 | 0.140538 | 7115.51 |

The number of batches matched:

```text
ceil(total_events / batch_size)
```

For example:

```text
1000 events / 64
→ 15 full batches
→ 1 final partial batch
→ 16 batches total
```

---

# Batching Throughput Improvement

Theoretical per-event processing cost was:

```text
Batch 1:
2.1 ms/event

Batch 4:
2.4 / 4
= 0.6 ms/event

Batch 16:
3.6 / 16
= 0.225 ms/event

Batch 64:
8.4 / 64
≈ 0.131 ms/event
```

Measured throughput increased from:

```text
448 events/s
```

to:

```text
7116 events/s
```

as batch size increased from 1 to 64.

This demonstrated that batching:

```text
amortizes fixed downstream overhead
        ↓
increases consumer service rate
        ↓
reduces queue buildup
        ↓
reduces producer backpressure
```

---

# Blocking Behavior with Batching

An important result was that larger batches produced:

```text
fewer
but longer
individual blocking periods
```

For example:

```text
Batch 1:
935 blocks
~2.23 ms each

Batch 64:
15 blocks
~8.03 ms each
```

The consumer with batch size 64 behaves approximately like:

```text
pop many events quickly
        ↓
free many queue slots
        ↓
producer runs in a burst
        ↓
consumer processes large batch
        ↓
producer waits
```

So block count alone is not sufficient to characterize backpressure.

More meaningful metrics include:

```text
total blocked time
average block duration
blocked push ratio
```

---

# Queue Capacity vs Batching

The experiments revealed an important distinction.

Increasing queue capacity:

```text
reduces producer blocking
```

but:

```text
does not improve throughput
```

because it only increases buffering.

Batching, however:

```text
reduces producer blocking
AND
increases throughput
AND
reduces runtime
```

because it improves consumer efficiency.

This can be summarized as:

```text
Larger queue
→ delays the bottleneck

Better batching
→ improves the bottleneck
```

---

# Slow Network / Collector Simulation

The final experiment studied how downstream slowdown propagates upstream.

The fixed configuration was:

```text
Total events:      1000
Queue capacity:    64
Batch size:        16
```

Only downstream delay changed:

```text
0 ms
1 ms
5 ms
10 ms
20 ms
```

This delay modeled the combined effect of:

```text
network transmission
collector processing
storage
database insertion
```

Measured results:

| Delay (ms) | Blocks | Block Ratio | Avg Block (ms) | Total Block (s) | Runtime (s) | Throughput (events/s) |
|---:|---:|---:|---:|---:|---:|---:|
| 0 | 4 | 0.4% | 0.25984 | 0.001039 | 0.008685 | 115136 |
| 1 | 69 | 6.9% | 0.943306 | 0.065088 | 0.074872 | 13356 |
| 5 | 60 | 6.0% | 4.96473 | 0.297884 | 0.327505 | 3053 |
| 10 | 62 | 6.2% | 9.50473 | 0.589293 | 0.643282 | 1555 |
| 20 | 60 | 6.0% | 19.5243 | 1.17146 | 1.27577 | 784 |

---

# Backpressure Propagation

The most important observation was:

```text
1 ms delay  → ~0.94 ms average block
5 ms delay  → ~4.96 ms average block
10 ms delay → ~9.50 ms average block
20 ms delay → ~19.52 ms average block
```

Average producer block duration closely tracked downstream delay.

This directly demonstrated:

```text
slow downstream
        ↓
consumer spends longer away from queue
        ↓
queue fills
        ↓
producer waits
```

The producer does not need to know whether the actual root cause is:

```text
network congestion

slow collector

database contention

slow storage
```

It only observes that the local bounded queue has no free capacity.

---

# Why Block Count Stayed Similar

For 5–20 ms downstream delays, block count remained around:

```text
~60 blocks
```

while each blocking period became much longer.

This happened because the batching consumer released multiple queue slots at once.

The pattern was approximately:

```text
consumer pops one batch
        ↓
producer fills newly available slots
        ↓
consumer processes downstream batch
        ↓
producer blocks
```

Increasing downstream delay therefore did not necessarily increase how many times the producer blocked.

Instead, it increased:

```text
how long each blocking episode lasted
```

This reinforced an important observability lesson:

> Block count alone is not enough to characterize backpressure.

---

# Maximum Queue Utilization

Maximum queue utilization reached:

```text
100%
```

for every downstream-delay configuration.

This did not mean that the queue remained full all the time.

It only meant:

```text
the queue reached full capacity at least once
```

Because the producer generated a very fast burst, even the zero-delay consumer could temporarily observe a full queue.

A more detailed production implementation could additionally measure:

```text
average queue occupancy
time spent at full capacity
queue occupancy distribution
```

---

# Main System-Level Conclusions

The experiments produced several important conclusions.

### 1. Backpressure throttles a fast producer

When the consumer cannot keep up:

```text
queue fills
↓
producer waits
↓
producer effective rate approaches consumer rate
```

---

### 2. Queue capacity controls burst absorption

A larger queue:

```text
reduces immediate producer blocking
```

but does not increase downstream throughput.

It delays backpressure rather than removing the bottleneck.

---

### 3. Overload policy defines the system's failure behavior

When the queue is full:

```text
Block
→ preserve all events
→ increase producer interference

DropNewest
→ preserve existing backlog
→ lose new events

DropOldest
→ preserve fresh events
→ lose old buffered events
```

The correct policy depends on profiling semantics.

---

### 4. Batching can improve the bottleneck itself

Batching amortizes fixed downstream cost.

Measured throughput increased from approximately:

```text
448 events/s
```

to:

```text
7116 events/s
```

when batch size increased from 1 to 64.

Unlike simply increasing queue capacity, batching improved the downstream service rate.

---

### 5. Larger batches create a throughput-latency tradeoff

Large batches provide:

```text
higher throughput
lower producer blocking
```

but may introduce:

```text
higher event waiting latency
```

especially when the arrival rate is low.

A production system would typically combine:

```text
maximum batch size
+
flush timeout
```

to avoid waiting indefinitely for a batch to fill.

---

### 6. Backpressure is an end-to-end property

A slowdown far downstream can eventually affect the producer:

```text
Slow Collector / Storage
        ↓
Slow network progress
        ↓
Slow local consumer
        ↓
Queue buildup
        ↓
Producer blocking
```

This remains true even when components are separated by:

```text
threads
queues
TCP
processes
different machines
```

---

# Profiling-System Interpretation

For a profiler, backpressure is especially important because excessive producer blocking can distort the application being measured.

The system therefore needs to balance:

```text
trace completeness

profiling overhead

memory usage

event freshness

throughput

latency
```

There is no universally optimal configuration.

For example:

```text
Exact trace collection
→ favor lossless behavior

Live telemetry
→ may favor DropOldest

Low-overhead sampling
→ may tolerate drops

Offline trace ingestion
→ may favor larger batches
```

---

# Final Day 6 Architecture

At the end of these experiments, the profiling pipeline can be understood as:

```text
Fast Profiling Producer
        ↓
BoundedQueue
        ↓
Batching Consumer
        ↓
Network / Collector
        ↓
Storage / Analysis
```

with several control mechanisms:

```text
queue capacity
overload policy
batch size
downstream service rate
```

The key engineering task is not simply to maximize one metric.

It is to choose a configuration that provides an acceptable balance among:

```text
application interference
data completeness
latency
memory
throughput
```

---

# Interview Summary

A concise interview answer for this work would be:

> **I built a bounded asynchronous profiling pipeline and used controlled experiments to study backpressure and scalability. I first measured how a slow consumer propagates backpressure to the producer and added precise instrumentation for blocking time and queue occupancy. Increasing queue capacity reduced producer blocking but did not improve throughput, showing that buffering only delays a downstream bottleneck.**
>
> **I then compared blocking, Drop-Newest, and Drop-Oldest policies to study the tradeoff between trace completeness and profiler overhead. After that, I added batching and showed that amortizing fixed downstream costs increased throughput from roughly 448 to over 7,100 events per second while reducing the blocked-push ratio from 93.5% to 1.5%. Finally, I simulated a slow network or collector and demonstrated that downstream latency propagates through the bounded queue back to the producer.**
>
> **The main conclusion was that backpressure is an end-to-end system property. Queue sizing, overload policy, batching, and downstream service capacity solve different parts of the problem and must be tuned according to whether the profiler prioritizes completeness, low interference, throughput, freshness, or latency.**